In [ ]:
import os, json, time, random, math, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T, models
from PIL import Image
from scipy.stats import chi2, binomtest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, cohen_kappa_score,
                             matthews_corrcoef, roc_auc_score, confusion_matrix)

# %% CONFIG ----------------------------------------------------------------
DATASET        = "hyperkvasir"                      # "hyperkvasir" or "kvasir_v2"
DATA_ROOT      = "/kaggle/input/the-hyper-kvasir-dataset"
# Kvasir-V2: DATA_ROOT = "/kaggle/input/kvasir-v2-a-gastrointestinal-tract-dataset"
HYPERKVASIR_ROOT = "/kaggle/input/the-hyper-kvasir-dataset"                       # used by "overlap"
KVASIR_V2_ROOT   = "/kaggle/input/kvasir-v2-a-gastrointestinal-tract-dataset"     # used by "overlap"
QUICK            = False     # True = 1-epoch pipeline test (about a few minutes). NOT for the paper. Set False for the real run.
PROCESSED        = True      # your processed dataset (train/test folders with *_aug_* files)
USE_AUG_IN_TRAIN = False     # True: originals + their augmented copies in training; False: originals only
PROC_VAL_FRAC    = 0.15      # validation share of the ORIGINAL training images
if PROCESSED:
    DATA_ROOT = "/kaggle/input/datasets/visheshtanwar26/processed-datatset"
OUT_DIR        = f"/kaggle/working/out_{DATASET}_{('proc_aug' if USE_AUG_IN_TRAIN else 'proc_noaug') if PROCESSED else 'plain'}{'_QUICK' if QUICK else ''}"
RUN            = ["main", "overlap"]                           # e.g. ["main", "ablation", "baselines", "sweeps", "overlap"]

BACKBONE       = "s"      # "s" -> EfficientNetV2-S, "m" -> EfficientNetV2-M. 
     
FINETUNE_EPOCHS = 0       # eval mode: >0 = first fine-tune the loaded model on the train split (validation split picks the best epoch), then test once
FINETUNE_LR    = 3e-5     # head/attention lr during that fine-tuning (backbone gets 0.1x)     
SPLIT_SEED     = 42
SEEDS          = [42, 43, 44]
ABL_SEEDS      = [42, 43, 44]                       # 7 variants x 3 seeds is long: run it over several sessions
VAL_FRAC, TEST_FRAC = 0.15, 0.15
IMG_SIZE, BATCH_SIZE = 224, 16
LR, WEIGHT_DECAY = 1e-4, 1e-5
HEAD_EPOCHS, FT_EPOCHS = 5, 35                      # total 40
LABEL_SMOOTHING = 0.05
EMBED_DIM, NUM_HEADS = 512, 8
WINDOW         = 4        # 224 px -> 7x7 map, zero-padded to 8x8 -> four 4x4 windows. WINDOW = 7 would be global attention
DROPOUT        = 0.3
NUM_WORKERS    = 0        # images are cached in RAM, workers are not needed
HASH_NEAR      = 4        # Hamming distance (of 64) counted as a near-duplicate
SWEEP_LRS      = [1e-3, 5e-4, 1e-4, 5e-5, 1e-5]
SWEEP_OPTS     = ["sgd", "rmsprop", "adam"]
if QUICK:
    HEAD_EPOCHS, FT_EPOCHS, SEEDS, ABL_SEEDS = 1, 0, [42], [42]
    print("QUICK MODE: 1 epoch, 1 seed. These numbers are only a pipeline test and must not be used in the paper.")

HYPERKVASIR_12 = {"bbps-0-1", "bbps-2-3", "cecum", "dyed-lifted-polyps", "dyed-resection-margins", "esophagitis",
                  "polyps", "pylorus", "retroflex-rectum", "retroflex-stomach",
                  "ulcerative-colitis", "z-line"}
KVASIR_V2_8 = {"dyed-lifted-polyps", "dyed-resection-margins", "esophagitis", "normal-cecum", "normal-pylorus",
               "normal-z-line", "polyps", "ulcerative-colitis"}
EXT = {".jpg", ".jpeg", ".png", ".bmp"}
device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)

# %% DATA ------------------------------------------------------------------
def norm_label(name):
    n = name.lower()
    if n.startswith("esophagitis"): return "esophagitis"
    if n.startswith("ulcerative-colitis"): return "ulcerative-colitis"
    return n

def list_images(root, keep):
    rows, seen = [], set()
    for dp, _, files in os.walk(root):
        lab = norm_label(Path(dp).name)
        if lab not in keep: continue
        for f in files:
            if Path(f).suffix.lower() in EXT and (lab, f) not in seen:
                seen.add((lab, f)); rows.append((os.path.join(dp, f), lab))
    return pd.DataFrame(rows, columns=["path", "label"])

def make_split_processed(root):
    """train/ may contain augmented copies named <id>_aug_<hash>.jpg; test/ holds originals.
    Validation is cut from the ORIGINAL training images (by id); augmented copies follow their original into train only."""
    rows = []
    for part in ("train", "test"):
        base = os.path.join(root, part)
        for dp, _, files in os.walk(base):
            if dp == base: continue
            lab = Path(dp).name
            for f in files:
                if Path(f).suffix.lower() in EXT:
                    stem = Path(f).stem
                    rows.append((os.path.join(dp, f), lab, part, "_aug_" in stem, stem.split("_aug_")[0]))
    df = pd.DataFrame(rows, columns=["path", "label", "part", "is_aug", "orig_id"])
    classes = sorted(df.label.unique()); df["y"] = df.label.map({c: i for i, c in enumerate(classes)})
    trn, te = df[df.part == "train"], df[df.part == "test"]
    orig = trn[~trn.is_aug].drop_duplicates("orig_id")
    shared = set(te.orig_id) & set(orig.orig_id)
    print(f"train: {len(orig)} originals + {int(trn.is_aug.sum())} augmented copies | test: {len(te)} | ids shared by train and test: {len(shared)}")
    if "is_aug" in te and te.is_aug.any(): print("WARNING: test folder contains augmented files:", int(te.is_aug.sum()))
    te = te[~te.orig_id.isin(orig.orig_id)]                       # drop any test image that also exists in train
    o_tr, o_va = train_test_split(orig, test_size=PROC_VAL_FRAC, stratify=orig.y, random_state=SPLIT_SEED)
    tr = trn[trn.orig_id.isin(o_tr.orig_id)]
    if not USE_AUG_IN_TRAIN: tr = tr[~tr.is_aug]
    va = trn[trn.orig_id.isin(o_va.orig_id) & ~trn.is_aug]
    print(f"originals per class in train/:\n{orig.label.value_counts().sort_index().to_string()}")
    print(f"augmented copies in training set: {int(tr.is_aug.sum())} | originals in training set: {int((~tr.is_aug).sum())}")
    return classes, df, tr, va, te

def make_split(df):
    classes = sorted(df.label.unique())
    df = df.copy(); df["y"] = df.label.map({c: i for i, c in enumerate(classes)})
    tv, te = train_test_split(df, test_size=TEST_FRAC, stratify=df.y, random_state=SPLIT_SEED)
    tr, va = train_test_split(tv, test_size=VAL_FRAC / (1 - TEST_FRAC), stratify=tv.y, random_state=SPLIT_SEED)
    return classes, tr, va, te

TF = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])      # no augmentation

# Images are decoded and resized ONCE and kept in RAM as uint8 (about 150 KB each). Same preprocessing as before
# (resize to 224 x 224, ImageNet normalisation, no augmentation), but an epoch no longer waits for JPEG decoding.
from concurrent.futures import ThreadPoolExecutor
_CACHE = {}
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def _load(p):
    im = Image.open(p)
    im.draft("RGB", (IMG_SIZE, IMG_SIZE))          # fast JPEG decode at reduced size (no-op for PNG)
    return np.array(im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)

def cache_images(paths):
    todo = [p for p in dict.fromkeys(paths) if p not in _CACHE]
    if not todo: return
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        for p, a in zip(todo, ex.map(_load, todo)): _CACHE[p] = a
    print(f"cached {len(todo)} images in {time.time() - t0:.0f}s")

class DS(Dataset):
    def __init__(self, d):
        self.p, self.y = d.path.tolist(), d.y.tolist(); cache_images(self.p)
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        x = torch.from_numpy(_CACHE[self.p[i]]).permute(2, 0, 1).float().div_(255)
        return (x - _MEAN) / _STD, self.y[i]

def loader(d, shuffle, seed=0):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(DS(d), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=True, generator=g if shuffle else None)

# %% MODELS ----------------------------------------------------------------
class SE(nn.Module):
    def __init__(self, c, r=16):
        super().__init__(); self.fc1, self.fc2 = nn.Linear(c, c // r), nn.Linear(c // r, c)
    def forward(self, x):
        s = torch.sigmoid(self.fc2(torch.relu(self.fc1(x.mean((2, 3))))))
        return x * s[:, :, None, None]

class WindowAttention(nn.Module):
    """Non-overlapping window self-attention; the map is zero-padded to a multiple of the window size."""
    def __init__(self, dim, heads, win):
        super().__init__(); self.win = win
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True); self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        B, C, H, W = x.shape; w = self.win
        ph, pw = (-H) % w, (-W) % w
        if ph or pw: x = nn.functional.pad(x, (0, pw, 0, ph))
        Hp, Wp = H + ph, W + pw
        t = x.permute(0, 2, 3, 1).reshape(B, Hp // w, w, Wp // w, w, C).permute(0, 1, 3, 2, 4, 5).reshape(-1, w * w, C)
        t = self.norm(t + self.attn(t, t, t, need_weights=False)[0])
        t = t.reshape(B, Hp // w, Wp // w, w, w, C).permute(0, 1, 3, 2, 4, 5).reshape(B, Hp, Wp, C)
        return t.permute(0, 3, 1, 2)[:, :, :H, :W]

class AxialAttentionH(nn.Module):
    """Attention along the height axis only."""
    def __init__(self, dim, heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True); self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        B, C, H, W = x.shape
        t = x.permute(0, 3, 2, 1).reshape(B * W, H, C)
        t = self.norm(t + self.attn(t, t, t, need_weights=False)[0])
        return t.reshape(B, W, H, C).permute(0, 3, 2, 1)

def effnet_features():
    if BACKBONE == "m":
        return models.efficientnet_v2_m(weights=models.EfficientNet_V2_M_Weights.DEFAULT).features
    return models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT).features

class HEASNet(nn.Module):
    """Variants: se / win / ax switches. With all three off this is the EfficientNetV2 base model."""
    def __init__(self, n_classes, se=True, win=True, ax=True):
        super().__init__()
        self.backbone = effnet_features()
        self.proj = nn.Conv2d(1280, EMBED_DIM, 1)
        self.se = SE(EMBED_DIM) if se else nn.Identity()
        self.win = WindowAttention(EMBED_DIM, NUM_HEADS, WINDOW) if win else nn.Identity()
        self.ax = AxialAttentionH(EMBED_DIM, NUM_HEADS) if ax else nn.Identity()
        self.drop, self.fc = nn.Dropout(DROPOUT), nn.Linear(EMBED_DIM, n_classes)
    def attn_modules(self): return [self.proj, self.se, self.win, self.ax]
    def forward(self, x):
        x = self.ax(self.win(self.se(self.proj(self.backbone(x)))))
        return self.fc(self.drop(x.mean((2, 3))))

VARIANTS = {"base": (False, False, False), "+SE": (True, False, False), "+Swin": (False, True, False),
            "+Swin+SE": (True, True, False), "+Axial": (False, False, True), "+SE+Axial": (True, False, True),
            "HEASNet": (True, True, True)}

class Baseline(nn.Module):
    def __init__(self, name, n_classes):
        super().__init__(); self.name = name
        if name == "ResNet50":
            m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT); m.fc = nn.Linear(2048, n_classes)
        elif name == "EfficientNetV2":
            m = (models.efficientnet_v2_m(weights=models.EfficientNet_V2_M_Weights.DEFAULT) if BACKBONE == "m"
                 else models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT))
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
        elif name == "ConvNeXt-T":
            m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
            m.classifier[2] = nn.Linear(m.classifier[2].in_features, n_classes)
        elif name == "ViT-B/16":
            m = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
            m.heads.head = nn.Linear(m.heads.head.in_features, n_classes)
        else: raise ValueError(name)
        self.m = m
    def forward(self, x): return self.m(x)
    def attn_modules(self): return []

# %% METRICS (pure numpy: no torch needed) ---------------------------------
def ece_score(probs, y, bins=15):
    conf, pred = probs.max(1), probs.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def mcnemar(y, pred_a, pred_b):
    a_ok, b_ok = pred_a == y, pred_b == y
    b, c = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())      # b: only A right, c: only B right
    n = b + c
    if n == 0: return dict(only_a=b, only_b=c, p_value=1.0, test="none")
    if n < 25: return dict(only_a=b, only_b=c, p_value=float(binomtest(b, n, 0.5).pvalue), test="exact binomial")
    stat = (abs(b - c) - 1) ** 2 / n
    return dict(only_a=b, only_b=c, p_value=float(chi2.sf(stat, 1)), test="chi-square (continuity corrected)")

def boot_ci(y, pred, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    v = [accuracy_score(y[i], pred[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def full_metrics(y, probs):
    K = probs.shape[1]; pred = probs.argmax(1)
    p, r, f1, sup = precision_recall_fscore_support(y, pred, labels=range(K), zero_division=0)
    mp, mr, mf, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    wf = precision_recall_fscore_support(y, pred, average="weighted", zero_division=0)[2]
    oh = np.eye(K)[y]
    return dict(acc=accuracy_score(y, pred), acc_ci=boot_ci(y, pred), macro_precision=mp, macro_recall=mr, macro_f1=mf,
                weighted_f1=wf, kappa=cohen_kappa_score(y, pred), mcc=matthews_corrcoef(y, pred),
                roc_auc_macro=roc_auc_score(y, probs, multi_class="ovr", average="macro", labels=list(range(K))),
                ece=ece_score(probs, y)), (p, r, f1, sup, [roc_auc_score(oh[:, k], probs[:, k]) for k in range(K)])

# %% TRAINING --------------------------------------------------------------
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def run_epoch(model, dl, crit, opt=None, scaler=None):
    train = opt is not None; model.train(train); tl = co = n = 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                out = model(x); loss = crit(out, y)
            if train:
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(y); co += (out.argmax(1) == y).sum().item(); n += len(y)
    return tl / n, co / n

def make_opt(name, groups, lr):
    ps = [dict(params=g["params"], lr=g["lr"]) for g in groups]
    if name == "adam": return torch.optim.Adam(ps, weight_decay=WEIGHT_DECAY)
    if name == "sgd": return torch.optim.SGD(ps, momentum=0.9, weight_decay=WEIGHT_DECAY)
    if name == "rmsprop": return torch.optim.RMSprop(ps, weight_decay=WEIGHT_DECAY)
    raise ValueError(name)

def train_model(model, dl_tr, dl_va, lr=LR, opt_name="adam", epochs=(HEAD_EPOCHS, FT_EPOCHS)):
    """Two-stage training for HEASNet variants; baselines are fine-tuned end-to-end for the same number of epochs."""
    model.to(device); crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    best, state, hist = -1.0, None, []
    def stage(groups, n_ep, tag):
        nonlocal best, state
        opt = make_opt(opt_name, groups, lr)
        for ep in range(n_ep):
            t_ep = time.time()
            trl, tra = run_epoch(model, dl_tr, crit, opt, scaler); vl, va = run_epoch(model, dl_va, crit)
            print(f"  stage {tag} epoch {ep + 1}/{n_ep}: train {tra:.4f} val {va:.4f} ({time.time() - t_ep:.0f}s)")
            hist.append(dict(stage=tag, epoch=len(hist) + 1, train_loss=trl, train_acc=tra, val_loss=vl, val_acc=va))
            if va > best: best, state = va, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if isinstance(model, HEASNet):
        for p in model.parameters(): p.requires_grad = False
        head = [p for m in model.attn_modules() for p in m.parameters()] + list(model.fc.parameters())
        for p in head: p.requires_grad = True
        stage([dict(params=head, lr=lr)], epochs[0], 1)                       # stage 1: new layers only
        for p in model.parameters(): p.requires_grad = True
        bb = list(model.backbone.parameters()); ids = {id(p) for p in bb}
        stage([dict(params=bb, lr=lr * 0.1), dict(params=[p for p in model.parameters() if id(p) not in ids], lr=lr)], epochs[1], 2)
    else:
        stage([dict(params=list(model.parameters()), lr=lr * 0.1)], sum(epochs), 2)
    model.load_state_dict(state)
    return best, pd.DataFrame(hist)

@torch.no_grad()
def predict(model, dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            o = model(x.to(device))
        P.append(torch.softmax(o.float(), 1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

def efficiency(model):
    out = dict(params_M=sum(p.numel() for p in model.parameters()) / 1e6, gmacs=None, latency_ms=None, peak_mem_MB=None)
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device); model.eval()
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as fc, torch.no_grad(): model(x)
        out["gmacs"] = fc.get_total_flops() / 2 / 1e9
    except Exception as e: print("GMACs not computed:", e)
    if device == "cuda":
        with torch.no_grad():
            for _ in range(10): model(x)
            torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.time()
            for _ in range(100): model(x)
            torch.cuda.synchronize()
        out["latency_ms"] = (time.time() - t0) / 100 * 1000; out["peak_mem_MB"] = torch.cuda.max_memory_allocated() / 2**20
    return out

def one_run(name, builder, seed, dls, **kw):
    """Train, pick best-val checkpoint, evaluate val and test, save predictions."""
    f = f"{OUT_DIR}/preds_{name.replace('/', '_')}_s{seed}.npz"
    if os.path.exists(f):
        z = np.load(f); return dict(val_acc=float(z["val_acc"]), probs=z["probs"], y=z["y"], cached=True)
    seed_all(seed); model = builder(); dl_tr, dl_va, dl_te = dls
    best, hist = train_model(model, dl_tr, dl_va, **kw)
    probs, y = predict(model, dl_te)
    np.savez(f, probs=probs, y=y, val_acc=best); hist.to_csv(f.replace("preds_", "history_").replace(".npz", ".csv"), index=False)
    if name == "HEASNet":      # best-validation weights, accuracies in the file name
        torch.save(model.state_dict(), f.replace("preds_", "model_").replace(".npz", f"_val{100 * best:.2f}_test{100 * accuracy_score(y, probs.argmax(1)):.2f}.pt"))
    res = dict(val_acc=best, probs=probs, y=y, cached=False, model=model)
    print(f"[{name} seed {seed}] val {best:.4f} test {accuracy_score(y, probs.argmax(1)):.4f}")
    return res

# %% SETUP ----------------------------------------------------------------
if PROCESSED:
    classes, df, tr, va, te = make_split_processed(DATA_ROOT)
else:
    df = list_images(DATA_ROOT, HYPERKVASIR_13 if DATASET == "hyperkvasir" else KVASIR_V2_8)
    classes, tr, va, te = make_split(df)
K = len(classes)
print("device:", device)
print(f"{DATASET}: {len(df)} images, {K} classes | train {len(tr)} val {len(va)} test {len(te)} | backbone EfficientNetV2-{BACKBONE.upper()}")
if not PROCESSED: print("expected totals: hyperkvasir 10,553; kvasir_v2 8,000")
pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")]).to_csv(f"{OUT_DIR}/splits.csv", index=False)
pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")]).pivot_table(
    index="label", columns="split", values="y", aggfunc="count").to_csv(f"{OUT_DIR}/class_split_counts.csv")
dls = (loader(tr, True), loader(va, False), loader(te, False))

# %% FULL REPORT FOR ONE MODEL: metrics, confusion matrices, ROC, efficiency, statistics, confidence, Grad-CAM ----
def _wilson(k, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def _plot_cm(cm, names, fn, title):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7.6, 6), dpi=200); im = ax.imshow(cm, cmap="Blues"); k = len(names)
    ax.set_xticks(range(k)); ax.set_yticks(range(k)); ax.set_xticklabels(names, rotation=60, ha="right", fontsize=7); ax.set_yticklabels(names, fontsize=7)
    for i in range(k):
        for j in range(k):
            if cm[i, j]: ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontsize=9); fig.colorbar(im, fraction=.046)
    fig.tight_layout(); fig.savefig(fn); plt.close(fig)

def efficiency_full(model):
    import copy
    out = dict(precision="FP32", gpu=(torch.cuda.get_device_name(0) if device == "cuda" else "cpu"))
    out["params_M"] = sum(p.numel() for p in model.parameters()) / 1e6
    out["weights_MB_fp32"] = sum(p.numel() for p in model.parameters()) * 4 / 2**20
    model.eval(); x1 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as fc, torch.no_grad(): model(x1)
        out["gmacs"] = fc.get_total_flops() / 2 / 1e9; out["gflops"] = fc.get_total_flops() / 1e9
    except Exception as e: print("FLOP counter failed:", e)
    if device == "cuda":
        with torch.no_grad():
            for _ in range(20): model(x1)
            torch.cuda.reset_peak_memory_stats(); ts = []
            for _ in range(200):
                torch.cuda.synchronize(); t0 = time.perf_counter(); model(x1); torch.cuda.synchronize(); ts.append((time.perf_counter() - t0) * 1000)
            out["gpu_latency_ms_mean"] = float(np.mean(ts)); out["gpu_latency_ms_sd"] = float(np.std(ts)); out["gpu_latency_ms_p95"] = float(np.percentile(ts, 95))
            out["gpu_peak_mem_MB_batch1"] = torch.cuda.max_memory_allocated() / 2**20
            xb = torch.randn(16, 3, IMG_SIZE, IMG_SIZE, device=device)
            for _ in range(5): model(xb)
            torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.perf_counter()
            for _ in range(30): model(xb)
            torch.cuda.synchronize(); dt = time.perf_counter() - t0
            out["gpu_throughput_img_per_s_batch16"] = 30 * 16 / dt; out["gpu_peak_mem_MB_batch16"] = torch.cuda.max_memory_allocated() / 2**20
    try:
        cm_ = copy.deepcopy(model).float().cpu().eval(); xc = torch.randn(1, 3, IMG_SIZE, IMG_SIZE); ts = []
        with torch.no_grad():
            for _ in range(3): cm_(xc)
            for _ in range(15):
                t0 = time.perf_counter(); cm_(xc); ts.append((time.perf_counter() - t0) * 1000)
        out["cpu_latency_ms_mean"] = float(np.mean(ts)); out["cpu_latency_ms_sd"] = float(np.std(ts)); del cm_
    except Exception as e: print("CPU latency skipped:", e)
    return out

def gradcam_report(model, dset, probs, y, outdir, per_class=2, n_err=12):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    import torch.nn.functional as F
    model.eval(); pred = probs.argmax(1); conf = probs.max(1); store = {}
    def hook(name):
        def f(m, i, o): store[name] = o
        return f
    hs = [getattr(model, n).register_forward_hook(hook(n)) for n in ("se", "win", "ax")]
    def maps(i):
        x, _ = dset[i]; x = x.unsqueeze(0).to(device); c = int(pred[i])
        with torch.enable_grad():
            store.clear(); out = model(x.float()); feat = store["ax"]
            g = torch.autograd.grad(out[0, c], feat)[0]
        cam = F.relu((g.mean((2, 3), keepdim=True) * feat).sum(1, keepdim=True)).detach()
        up = lambda t: F.interpolate(t, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
        nz = lambda a: (a - a.min()) / (a.max() - a.min() + 1e-8)
        br = {n: nz(up(store[n].detach().abs().mean(1, keepdim=True))) for n in ("se", "win", "ax")}
        img = (x[0].cpu() * _STD + _MEAN).clamp(0, 1).permute(1, 2, 0).numpy()
        return img, nz(up(cam)), br
    rng = np.random.RandomState(0); K_ = probs.shape[1]; pick = []
    for c in range(K_):
        ok = np.where((y == c) & (pred == c))[0]
        if len(ok): pick += [(int(i), "correct") for i in rng.choice(ok, min(per_class, len(ok)), replace=False)]
    bad = np.where(pred != y)[0]; bad = bad[np.argsort(-conf[bad])][:n_err]; pick_err = [(int(i), "error") for i in bad]
    corr = {n: [[] for _ in range(K_)] for n in ("se", "win", "ax")}
    def draw(items, fn, cols=4):
        rows = int(math.ceil(len(items) / cols)); fig, axs = plt.subplots(rows, cols * 2, figsize=(cols * 3.2, rows * 1.7), dpi=150); axs = np.atleast_2d(axs)
        for a in axs.ravel(): a.axis("off")
        for k, (i, kind) in enumerate(items):
            img, cam, br = maps(i); r, c0 = divmod(k, cols)
            for n in br:
                cc = np.corrcoef(cam.ravel(), br[n].ravel())[0, 1]; corr[n][int(pred[i])].append(0.0 if np.isnan(cc) else cc)
            axs[r, c0 * 2].imshow(img); axs[r, c0 * 2 + 1].imshow(img); axs[r, c0 * 2 + 1].imshow(cam, cmap="jet", alpha=.45)
            axs[r, c0 * 2].set_title(f"T:{classes[y[i]][:12]}\nP:{classes[pred[i]][:12]} {conf[i]:.2f}", fontsize=5.5, color=("green" if kind == "correct" else "red"))
        fig.tight_layout(); fig.savefig(fn); plt.close(fig)
    draw(pick, f"{outdir}/gradcam_correct_per_class.png"); draw(pick_err, f"{outdir}/gradcam_misclassified.png")
    for h in hs: h.remove()
    rows = [dict(**{"class": classes[c]}, **{f"corr_{n}": (float(np.mean(corr[n][c])) if corr[n][c] else float("nan")) for n in corr}, n_samples=len(corr["se"][c])) for c in range(K_)]
    pd.DataFrame(rows).to_csv(f"{outdir}/gradcam_attention_correlation.csv", index=False)

def full_report(model, outdir, dls_, MAIN_OUT_DIR=None):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve, auc, log_loss
    os.makedirs(outdir, exist_ok=True); names = [c.capitalize() for c in classes]; Kc = len(classes)
    sets = {"train": loader(tr, False), "val": dls_[1], "test": dls_[2]}; res = {}
    t0 = time.perf_counter()
    for s, dl in sets.items():
        P, Y = predict(model, dl); m, pc = full_metrics(Y, P); res[s] = (P, Y, m, pc)
    print(f"prediction time for {len(tr) + len(va) + len(te)} images: {time.perf_counter() - t0:.1f}s")
    rows = []
    for s, (P, Y, m, pc) in res.items():
        cm = confusion_matrix(Y, P.argmax(1), labels=range(Kc))
        pd.DataFrame(cm, index=classes, columns=classes).to_csv(f"{outdir}/confusion_matrix_{s}.csv"); _plot_cm(cm, names, f"{outdir}/fig_confusion_{s}.png", f"{s} set (n={len(Y)})")
        spec = [(cm.sum() - cm[k].sum() - cm[:, k].sum() + cm[k, k]) / max(cm.sum() - cm[k].sum(), 1) for k in range(Kc)]
        pd.DataFrame(dict(**{"class": classes}, precision=pc[0], recall_sensitivity=pc[1], specificity=spec, f1=pc[2], support=pc[3], roc_auc=pc[4])).to_csv(f"{outdir}/classification_report_{s}.csv", index=False)
        rows.append(dict(split=s, n=len(Y), **{k: v for k, v in m.items() if k != "acc_ci"}, acc_ci_lo=m["acc_ci"][0], acc_ci_hi=m["acc_ci"][1],
                         brier=float(((P - np.eye(Kc)[Y]) ** 2).sum(1).mean()), nll=float(log_loss(Y, P, labels=list(range(Kc))))))
    pd.DataFrame(rows).to_csv(f"{outdir}/split_metrics.csv", index=False)
    P, Y, m, pc = res["test"]; pred = P.argmax(1); conf = P.max(1); ok = pred == Y
    fig, ax = plt.subplots(figsize=(6.6, 5.2), dpi=200); oh = np.eye(Kc)[Y]
    for c in range(Kc):
        fpr, tpr, _ = roc_curve(oh[:, c], P[:, c]); ax.plot(fpr, tpr, lw=1.1, label=f"{names[c]} ({auc(fpr, tpr):.3f})")
    fpr, tpr, _ = roc_curve(oh.ravel(), P.ravel()); ax.plot(fpr, tpr, "k--", lw=1.5, label=f"micro ({auc(fpr, tpr):.3f})"); ax.plot([0, 1], [0, 1], "k:", lw=.8)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend(fontsize=6, loc="lower right"); fig.tight_layout(); fig.savefig(f"{outdir}/fig_roc_test.png"); plt.close(fig)
    rng = np.random.RandomState(0); boots = []
    for _ in range(500):
        i = rng.randint(0, len(Y), len(Y)); yb, pb = Y[i], P[i]
        try: auc_b = roc_auc_score(yb, pb, multi_class="ovr", average="macro", labels=list(range(Kc)))
        except Exception: auc_b = np.nan
        pr = pb.argmax(1); boots.append([accuracy_score(yb, pr), precision_recall_fscore_support(yb, pr, average="macro", zero_division=0)[2], cohen_kappa_score(yb, pr), matthews_corrcoef(yb, pr), auc_b])
    boots = np.array(boots); est = [m["acc"], m["macro_f1"], m["kappa"], m["mcc"], m["roc_auc_macro"]]
    pd.DataFrame([dict(metric=n, estimate=e, ci_lo=np.nanpercentile(boots[:, k], 2.5), ci_hi=np.nanpercentile(boots[:, k], 97.5), bootstrap_sd=np.nanstd(boots[:, k]))
                  for k, (n, e) in enumerate(zip(["accuracy", "macro_f1", "kappa", "mcc", "macro_roc_auc"], est))]).to_csv(f"{outdir}/bootstrap_ci_test.csv", index=False)
    maj = np.bincount(Y).max() / len(Y); k_ok = int(ok.sum())
    stat = dict(n_test=len(Y), correct=k_ok, majority_class_rate=float(maj), binom_p_vs_majority=float(binomtest(k_ok, len(Y), maj, alternative="greater").pvalue),
                binom_p_vs_chance=float(binomtest(k_ok, len(Y), 1 / Kc, alternative="greater").pvalue), kappa=m["kappa"], mcc=m["mcc"])
    pd.DataFrame([dict(**{"class": classes[k]}, recall=pc[1][k], ci_lo=_wilson(int(round(pc[1][k] * pc[3][k])), int(pc[3][k]))[0], ci_hi=_wilson(int(round(pc[1][k] * pc[3][k])), int(pc[3][k]))[1], support=int(pc[3][k])) for k in range(Kc)]).to_csv(f"{outdir}/per_class_recall_ci_test.csv", index=False)
    import glob; mc = []
    for d_ in {x for x in (OUT_DIR, MAIN_OUT_DIR) if x}:
        for f_ in glob.glob(f"{d_}/preds_*_s{SEEDS[0]}.npz"):
            nm = os.path.basename(f_)[6:-4]
            if d_ == outdir or nm.startswith("HEASNet"): continue
            z = np.load(f_)
            if len(z["y"]) == len(Y) and (z["y"] == Y).all():
                r_ = mcnemar(Y, pred, z["probs"].argmax(1)); r_["compared_with"] = nm; r_["acc_other"] = float((z["probs"].argmax(1) == Y).mean()); mc.append(r_)
    if mc: pd.DataFrame(mc).to_csv(f"{outdir}/mcnemar_vs_other_models.csv", index=False)
    json.dump(stat, open(f"{outdir}/statistical_tests.json", "w"), indent=2, default=float)
    ent = -(P * np.log(P + 1e-12)).sum(1); edges = np.linspace(0, 1, 16); gaps, rel = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        b = (conf > lo) & (conf <= hi)
        if b.any(): gaps.append(abs(ok[b].mean() - conf[b].mean())); rel.append((conf[b].mean(), ok[b].mean(), int(b.sum())))
    conf_sum = dict(mean_conf_all=float(conf.mean()), mean_conf_correct=float(conf[ok].mean()), mean_conf_wrong=float(conf[~ok].mean()) if (~ok).any() else None,
                    mean_entropy_correct=float(ent[ok].mean()), mean_entropy_wrong=float(ent[~ok].mean()) if (~ok).any() else None, ece_15bins=m["ece"],
                    mce_15bins=float(max(gaps)), brier=rows[2]["brier"], nll=rows[2]["nll"])
    json.dump(conf_sum, open(f"{outdir}/confidence_summary.json", "w"), indent=2)
    fig, ax = plt.subplots(figsize=(4.6, 4.4), dpi=200); ax.plot([0, 1], [0, 1], "k:"); ax.plot([r[0] for r in rel], [r[1] for r in rel], "o-", color="#1f6f8b")
    ax.set_xlabel("Mean confidence"); ax.set_ylabel("Accuracy"); ax.set_title(f"Reliability diagram (ECE {m['ece']:.3f})", fontsize=9); fig.tight_layout(); fig.savefig(f"{outdir}/fig_reliability.png"); plt.close(fig)
    pd.DataFrame([dict(conf_threshold=t, coverage=float((conf >= t).mean()), accuracy_on_covered=float(ok[conf >= t].mean()) if (conf >= t).any() else float("nan"))
                  for t in (0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99)]).to_csv(f"{outdir}/selective_accuracy_test.csv", index=False)
    eff = efficiency_full(model); json.dump(eff, open(f"{outdir}/efficiency_full.json", "w"), indent=2, default=float); print(json.dumps(eff, indent=2, default=float))
    try: gradcam_report(model, dls_[2].dataset, P, Y, outdir)
    except Exception as e: print("Grad-CAM skipped:", repr(e))
    print("test accuracy", round(m["acc"], 4), "| kappa", round(m["kappa"], 4), "| mcc", round(m["mcc"], 4), "| macro AUC", round(m["roc_auc_macro"], 4))
    import shutil; z = shutil.make_archive(outdir.rstrip("/") + "_download", "zip", outdir); print("ZIP with all tables and figures (download this):", z)

# %% EVAL ONLY: use a model you already trained ------------------------------------
if "eval" in RUN:
    assert CKPT and os.path.exists(CKPT), "set CKPT to the path of your trained weights"
    MAIN_OUT_DIR = OUT_DIR; OUT_DIR = OUT_DIR + "_eval"; os.makedirs(OUT_DIR, exist_ok=True)
    try:
        ck = torch.load(CKPT, map_location="cpu")
    except Exception:
        ck = torch.load(CKPT, map_location="cpu", weights_only=False)   # whole-model pickle (your own file)
    if isinstance(ck, nn.Module): model = ck.to(device)
    else:
        if isinstance(ck, dict):
            for k in ("state_dict", "model_state_dict", "model", "net"):
                if k in ck and isinstance(ck[k], dict): ck = ck[k]; break
        ck = {k.replace("module.", ""): v for k, v in ck.items()}
        model, found = None, None
        print(f"checkpoint: {len(ck)} tensors; first keys:")
        for k_, v_ in list(ck.items())[:6]: print("   ", k_, tuple(v_.shape))
        print("    ...  last keys:")
        for k_, v_ in list(ck.items())[-4:]: print("   ", k_, tuple(v_.shape))
        fcw = [v_ for k_, v_ in ck.items() if k_.endswith("fc.weight") or k_.endswith("classifier.weight")]
        if fcw: print(f"classes in checkpoint: {fcw[-1].shape[0]} | classes in current data: {K}")
        errs = []
        for bb in dict.fromkeys([BACKBONE, "s", "m"]):          # try the settings the checkpoint may have been trained with
            for wn in dict.fromkeys([CKPT_WINDOW, 7, 4]):
                BACKBONE, WINDOW = bb, wn
                try:
                    cand = HEASNet(K); res = cand.load_state_dict(ck, strict=False)
                    if not res.missing_keys and not res.unexpected_keys: model, found = cand.to(device), (bb, wn); break
                    errs.append((bb, wn, "missing", res.missing_keys[:3], "unexpected", res.unexpected_keys[:3]))
                except Exception as e: errs.append((bb, wn, str(e)[:300]))
            if model is not None: break
        if model is None:
            for e_ in errs[:3]: print("tried:", e_)
            raise SystemExit("The checkpoint does not fit this model code/class set. Paste the 'checkpoint:' and 'tried:' lines above back to me.")
        print(f"checkpoint matched: backbone EfficientNetV2-{found[0].upper()}, window {found[1]}")
    if FINETUNE_EPOCHS > 0:
        print(f"fine-tuning the loaded model for {FINETUNE_EPOCHS} epochs on the train split (validation split selects the best epoch)")
        _t = time.time(); _b, _h = train_model(model, dls[0], dls[1], lr=FINETUNE_LR, epochs=(0, FINETUNE_EPOCHS))
        _h.to_csv(f"{OUT_DIR}/history_finetune.csv", index=False)
        print(f"fine-tuning done: best val {_b:.4f}, {(time.time() - _t) / FINETUNE_EPOCHS:.0f}s per epoch")
    print("!! The result is valid only if this model was NOT trained on any image of the test folder. "
          "A model from the first run (which pooled train and test) must not be evaluated on the test folder.")
    probs, y = predict(model, dls[2]); m, pc = full_metrics(y, probs)
    pv, yv = predict(model, dls[1]); val_acc = accuracy_score(yv, pv.argmax(1))
    ptr, ytr = predict(model, loader(tr, False)); print(f"accuracy of this checkpoint: train {accuracy_score(ytr, ptr.argmax(1)):.4f} | val {val_acc:.4f} | test {m['acc']:.4f}")
    np.savez(f"{OUT_DIR}/preds_HEASNet_s{SEEDS[0]}.npz", probs=probs, y=y, val_acc=val_acc)
    pd.DataFrame(dict(**{"class": classes}, precision=pc[0], recall=pc[1], f1=pc[2], support=pc[3], roc_auc=pc[4])).to_csv(
        f"{OUT_DIR}/test_classification_report.csv", index=False)
    pd.DataFrame(confusion_matrix(y, probs.argmax(1)), index=classes, columns=classes).to_csv(f"{OUT_DIR}/confusion_matrix.csv")
    eff = efficiency(model)
    summary = dict(mode="evaluation of a supplied checkpoint (no training)", checkpoint=CKPT, dataset=DATASET, n_classes=K,
                   n_train=len(tr), n_val=len(va), n_test=len(te), val_acc=val_acc,
                   test=dict(**{k: v for k, v in m.items() if k != "acc_ci"}, acc_ci_lo=m["acc_ci"][0], acc_ci_hi=m["acc_ci"][1]), efficiency=eff)
    json.dump(summary, open(f"{OUT_DIR}/test_metrics.json", "w"), indent=2, default=float); print(json.dumps(summary, indent=2, default=float))
    _tag = f"val{100 * val_acc:.2f}_test{100 * m['acc']:.2f}"
    torch.save(model.state_dict(), f"{OUT_DIR}/best_model_{'finetuned_' if FINETUNE_EPOCHS > 0 else ''}{_tag}.pth")   # the model that was evaluated, accuracies in the name
    print("saved:", f"{OUT_DIR}/best_model_{'finetuned_' if FINETUNE_EPOCHS > 0 else ''}{_tag}.pth")
    full_report(model, OUT_DIR, dls, MAIN_OUT_DIR)

# %% MAIN -------------------------------------------------------------------
if "main" in RUN:
    accs, rows = [], []
    for s in SEEDS:
        r = one_run("HEASNet", lambda: HEASNet(K), s, dls); m, pc = full_metrics(r["y"], r["probs"]); accs.append(m["acc"])
        rows.append(dict(seed=s, val_acc=r["val_acc"], **{k: v for k, v in m.items() if k != "acc_ci"}, acc_ci_lo=m["acc_ci"][0], acc_ci_hi=m["acc_ci"][1]))
        if s == SEEDS[0]:
            pd.DataFrame(dict(**{"class": classes}, precision=pc[0], recall=pc[1], f1=pc[2], support=pc[3], roc_auc=pc[4])).to_csv(
                f"{OUT_DIR}/test_classification_report.csv", index=False)
            pd.DataFrame(confusion_matrix(r["y"], r["probs"].argmax(1)), index=classes, columns=classes).to_csv(f"{OUT_DIR}/confusion_matrix.csv")
            eff = efficiency(r["model"] if "model" in r else HEASNet(K).to(device))
    per_seed = pd.DataFrame(rows); per_seed.to_csv(f"{OUT_DIR}/main_per_seed.csv", index=False)
    summary = dict(dataset=DATASET, n_classes=K, n_train=len(tr), n_val=len(va), n_test=len(te), backbone=f"EfficientNetV2-{BACKBONE.upper()}",
                   seeds=SEEDS, test_accuracy_mean=float(np.mean(accs)), test_accuracy_sd=float(np.std(accs, ddof=1)) if len(accs) > 1 else None,
                   first_seed=rows[0], efficiency=eff, augmentation=("offline copies in training only" if (PROCESSED and USE_AUG_IN_TRAIN) else "none"), epochs_total=HEAD_EPOCHS + FT_EPOCHS)
    json.dump(summary, open(f"{OUT_DIR}/test_metrics.json", "w"), indent=2, default=float)
    print(json.dumps(summary, indent=2, default=float))

# %% ABLATION ----------------------------------------------------------------
if "ablation" in RUN:
    rows = []
    for name, (se, win, ax) in VARIANTS.items():
        accs = []
        for s in ABL_SEEDS:
            r = one_run("HEASNet" if name == "HEASNet" else f"abl_{name}", lambda: HEASNet(K, se, win, ax), s, dls); accs.append(accuracy_score(r["y"], r["probs"].argmax(1)))
        e = efficiency(HEASNet(K, se, win, ax).to(device))
        rows.append(dict(variant=name, EfficientNetV2="Yes", Swin=("Yes" if win else "No"), Axial=("Yes" if ax else "No"), SE=("Yes" if se else "No"),
                         params_M=e["params_M"], gmacs=e["gmacs"], acc_mean=100 * np.mean(accs), acc_sd=100 * (np.std(accs, ddof=1) if len(accs) > 1 else float("nan")), n_seeds=len(accs)))
        pd.DataFrame(rows).to_csv(f"{OUT_DIR}/ablation_results.csv", index=False)
    print(pd.DataFrame(rows).round(3).to_string())

# %% BASELINES + McNEMAR --------------------------------------------------------
if "baselines" in RUN:
    rows, best_base, best_acc = [], None, -1
    for name in ["ResNet50", "EfficientNetV2", "ConvNeXt-T", "ViT-B/16"]:
        accs = []
        for s in SEEDS:
            r = one_run(f"base_{name}", lambda: Baseline(name, K), s, dls); accs.append(accuracy_score(r["y"], r["probs"].argmax(1)))
            if s == SEEDS[0] and np.mean(accs) > best_acc: best_acc, best_base = np.mean(accs), name
        e = efficiency(Baseline(name, K).to(device))
        rows.append(dict(model=name, acc_mean=100 * np.mean(accs), acc_sd=100 * (np.std(accs, ddof=1) if len(accs) > 1 else float("nan")), **e))
    pd.DataFrame(rows).to_csv(f"{OUT_DIR}/baselines.csv", index=False); print(pd.DataFrame(rows).round(3).to_string())
    a = np.load(f"{OUT_DIR}/preds_HEASNet_s{SEEDS[0]}.npz"); b = np.load(f"{OUT_DIR}/preds_base_{best_base}_s{SEEDS[0]}.npz")
    res = mcnemar(a["y"], a["probs"].argmax(1), b["probs"].argmax(1)); res["strongest_baseline"] = best_base
    pd.DataFrame([res]).to_csv(f"{OUT_DIR}/mcnemar.csv", index=False); print(res)

# %% SWEEPS (validation accuracy only; Tables 9 and 10) --------------------------
if "sweeps" in RUN:
    rows = []
    for lr in SWEEP_LRS:
        r = one_run("HEASNet" if lr == LR else f"sweep_adam_lr{lr:g}", lambda: HEASNet(K), SEEDS[0], dls, lr=lr)
        rows.append(dict(kind="learning_rate", setting=f"{lr:g}", val_acc=100 * r["val_acc"]))
    for o in SWEEP_OPTS:
        if o == "adam": rows.append(dict(kind="optimizer", setting="adam", val_acc=[x for x in rows if x["setting"] == f"{LR:g}"][0]["val_acc"])); continue
        r = one_run(f"sweep_{o}", lambda: HEASNet(K), SEEDS[0], dls, opt_name=o)
        rows.append(dict(kind="optimizer", setting=o, val_acc=100 * r["val_acc"]))
    pd.DataFrame(rows).to_csv(f"{OUT_DIR}/sweeps.csv", index=False); print(pd.DataFrame(rows).round(2).to_string())
    print("NOTE: Tables 9-10 report validation accuracy only; the 1e-4 row and the Adam row are the main seed-42 run.")

# %% OVERLAP / NEAR-DUPLICATE CHECKS --------------------------------------------
_POP = np.array([bin(i).count("1") for i in range(65536)], dtype=np.uint8)
def popcount64(x): return _POP[x.view(np.uint16).reshape(-1, 4)].sum(1)

def dhash(path, size=8):
    im = Image.open(path); im.draft("L", (4 * size, 4 * size))
    im = im.convert("L").resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(im, dtype=np.int16); bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

def min_dist(queries, ref):
    q, r = np.array(queries, dtype=np.uint64), np.array(ref, dtype=np.uint64)
    return np.array([popcount64(r ^ h).min() for h in q])

if "overlap" in RUN:
    h = {s: [dhash(p) for p in part.path] for s, part in (("train", tr), ("val", va), ("test", te))}
    out = {"dataset": DATASET, "hash": "dHash 64-bit", "near_threshold_bits": HASH_NEAR}
    for s in ("val", "test"):
        d_ = min_dist(h[s], h["train"])
        out[f"{s}_exact_duplicates_in_train"] = int((d_ == 0).sum()); out[f"{s}_near_duplicates_in_train"] = int((d_ <= HASH_NEAR).sum()); out[f"{s}_n"] = len(h[s])
    if DATASET == "kvasir_v2" and os.path.isdir(HYPERKVASIR_ROOT):
        hk = list_images(HYPERKVASIR_ROOT, HYPERKVASIR_13); hh = [dhash(p) for p in hk.path]
        allv = [dhash(p) for p in df.path]; d_ = min_dist(allv, hh)
        out["kvasir_v2_images"] = len(allv); out["kvasir_v2_exact_in_hyperkvasir"] = int((d_ == 0).sum()); out["kvasir_v2_near_in_hyperkvasir"] = int((d_ <= HASH_NEAR).sum())
    json.dump(out, open(f"{OUT_DIR}/overlap_check.json", "w"), indent=2); print(json.dumps(out, indent=2))
    # accuracy of the main model on test images WITHOUT a near-duplicate in the training set
    dt = min_dist(h["test"], h["train"]); keep = dt > HASH_NEAR; rows_dd = []
    for sd in SEEDS:
        f_ = f"{OUT_DIR}/preds_HEASNet_s{sd}.npz"
        if os.path.exists(f_):
            z = np.load(f_); ok = z["probs"].argmax(1) == z["y"]
            rows_dd.append(dict(seed=sd, n_all=len(ok), acc_all=100 * ok.mean(), n_no_dup=int(keep.sum()), acc_no_dup=100 * ok[keep].mean(),
                                n_dup=int((~keep).sum()), acc_dup=100 * ok[~keep].mean()))
    if rows_dd: pd.DataFrame(rows_dd).to_csv(f"{OUT_DIR}/dedup_accuracy.csv", index=False)

# %% FIGURES (seed SEEDS[0]) -> PNG files in OUT_DIR ---------------------------------
def make_figures():
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve, auc
    s0 = SEEDS[0]; hf = f"{OUT_DIR}/history_HEASNet_s{s0}.csv"; pf = f"{OUT_DIR}/preds_HEASNet_s{s0}.npz"
    if os.path.exists(hf):
        hh = pd.read_csv(hf); x = np.arange(1, len(hh) + 1)
        for key, lab, fn in (("loss", "Loss", "fig2_loss.png"), ("acc", "Accuracy", "fig3_accuracy.png")):
            fig, ax = plt.subplots(figsize=(6.4, 4.2), dpi=200)
            ax.plot(x, hh[f"train_{key}"], label="Training"); ax.plot(x, hh[f"val_{key}"], label="Validation")
            ax.axvline(HEAD_EPOCHS + 0.5, color="gray", ls=":"); ax.set_xlabel("Epoch"); ax.set_ylabel(lab); ax.grid(alpha=.25); ax.legend()
            fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{fn}"); plt.close(fig)
    if os.path.exists(pf):
        z = np.load(pf); P, Y = z["probs"], z["y"]
        cm = confusion_matrix(Y, P.argmax(1)); fig, ax = plt.subplots(figsize=(7.6, 6), dpi=200); im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(K)); ax.set_yticks(range(K)); ax.set_xticklabels(classes, rotation=60, ha="right", fontsize=7); ax.set_yticklabels(classes, fontsize=7)
        for i in range(K):
            for j in range(K):
                if cm[i, j]: ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > cm.max() / 2 else "black")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); fig.colorbar(im, fraction=.046); fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig4_confusion.png"); plt.close(fig)
        fig, ax = plt.subplots(figsize=(6.4, 5), dpi=200)
        for c in range(K):
            fpr, tpr, _ = roc_curve((Y == c).astype(int), P[:, c]); ax.plot(fpr, tpr, lw=1.2, label=f"{classes[c]} (AUC {auc(fpr, tpr):.3f})")
        ax.plot([0, 1], [0, 1], "k:", lw=1); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend(fontsize=6, loc="lower right")
        fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig5_roc.png"); plt.close(fig)
        print("figures written to", OUT_DIR)
if "main" in RUN or "eval" in RUN:
    try: make_figures()
    except Exception as e: print("figures skipped:", e)

# %% ONE BLOCK TO COPY BACK ---------------------------------------------------
def print_results_block():
    """Prints every result file in one block. Copy everything between the two ##### lines and send it back."""
    print("#" * 20, "COPY FROM HERE", "#" * 20)
    if QUICK: print("!!! QUICK MODE (1 epoch): do NOT use these numbers in the paper !!!")
    print(f"dataset={DATASET} processed={PROCESSED} aug_in_train={USE_AUG_IN_TRAIN} backbone=EfficientNetV2-{BACKBONE.upper()} window={WINDOW} seeds={SEEDS}")
    print(f"n_classes={K} n_train={len(tr)} n_val={len(va)} n_test={len(te)}")
    print("classes:", classes)
    for name in ["class_split_counts.csv", "test_metrics.json", "main_per_seed.csv", "test_classification_report.csv", "confusion_matrix.csv",
                 "ablation_results.csv", "baselines.csv", "mcnemar.csv", "sweeps.csv", "overlap_check.json", "dedup_accuracy.csv", "split_metrics.csv", "classification_report_train.csv", "classification_report_val.csv",
                 "confusion_matrix_train.csv", "confusion_matrix_val.csv", "bootstrap_ci_test.csv", "per_class_recall_ci_test.csv", "statistical_tests.json",
                 "mcnemar_vs_other_models.csv", "confidence_summary.json", "selective_accuracy_test.csv", "efficiency_full.json", "gradcam_attention_correlation.csv"]:
        f = f"{OUT_DIR}/{name}"
        if os.path.exists(f):
            print(f"\n=== {name} ==="); print(open(f).read().strip())
    h = f"{OUT_DIR}/history_HEASNet_s{SEEDS[0]}.csv"
    if os.path.exists(h):
        hh = pd.read_csv(h); b = hh.loc[hh.val_acc.idxmax()]
        print(f"\n=== training summary (seed {SEEDS[0]}) ===")
        print(f"best val acc {b.val_acc:.4f} at epoch {int(b.epoch)}; final epoch train acc {hh.train_acc.iloc[-1]:.4f}, val acc {hh.val_acc.iloc[-1]:.4f}, train loss {hh.train_loss.iloc[-1]:.4f}, val loss {hh.val_loss.iloc[-1]:.4f}")
        print("history.csv (every 5th epoch):"); print(hh.iloc[4::5].round(4).to_csv(index=False).strip())
    print("#" * 20, "COPY TO HERE", "#" * 20)

print("Done. Results in", OUT_DIR)
print_results_block()

In [ ]:
import os
import shutil

# Kaggle output directory
SOURCE_DIR = "/kaggle/working/out_hyperkvasir_proc_noaug"

# ZIP file location
ZIP_PATH = "/kaggle/working/all_outputs.zip"

# Create ZIP containing all output files and folders
shutil.make_archive(
    "/kaggle/all_outputs",
    "zip",
    SOURCE_DIR
)

# Move ZIP to Kaggle working directory
shutil.move("/kaggle/all_outputs.zip", ZIP_PATH)

print("ZIP created successfully!")
print("ZIP location:", ZIP_PATH)
print("ZIP size: {:.2f} MB".format(
    os.path.getsize(ZIP_PATH) / (1024**2)
))

In [ ]:
import os
import shutil

# Kaggle output directory
SOURCE_DIR = "/kaggle/working/out_hyperkvasir_proc_noaug_eval"

# ZIP file location
ZIP_PATH = "/kaggle/working/out_hyperkvasir_proc_noaug_eval.zip"

# Create ZIP containing all output files and folders
shutil.make_archive(
    "/kaggle/out_hyperkvasir_proc_noaug_eval",
    "zip",
    SOURCE_DIR
)

# Move ZIP to Kaggle working directory
shutil.move("/kaggle/out_hyperkvasir_proc_noaug_eval.zip", ZIP_PATH)

print("ZIP created successfully!")
print("ZIP location:", ZIP_PATH)
print("ZIP size: {:.2f} MB".format(
    os.path.getsize(ZIP_PATH) / (1024**2)
))

In [ ]:
# %%
import os, json, time, random, math, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T, models
from PIL import Image
from scipy.stats import chi2, binomtest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, cohen_kappa_score,
                             matthews_corrcoef, roc_auc_score, confusion_matrix)

# %% CONFIG ----------------------------------------------------------------
DATASET        = "kvasir_v2"
KVASIR_V2_DATA = "/kaggle/input/datasets/subhajeetdas/kvasir-v2-gastrointestinal-image-dataset/KVASIR V2"   # the ONLY dataset this file reads
DATA_ROOT      = KVASIR_V2_DATA
PROCESSED      = False       # Kvasir-V2 has one folder per class: stratified 70/15/15 split, no augmentation
USE_AUG_IN_TRAIN = False
PROC_VAL_FRAC  = 0.15
QUICK          = False       # True = 1-epoch pipeline test only
HYPERKVASIR_ROOT = "/nonexistent"                                                   # not used
HK_PROCESSED   = "/kaggle/input/datasets/visheshtanwar26/processed-datatset"        # OPTIONAL: only used by "overlap" if you also attached it; skipped otherwise
KVASIR_V2_ROOT = KVASIR_V2_DATA
def _find_kvasir_root():
    """Find the folder that contains the Kvasir-V2 class folders, wherever Kaggle mounted it."""
    if os.path.isdir(KVASIR_V2_DATA) and any(os.path.isdir(os.path.join(KVASIR_V2_DATA, d)) for d in os.listdir(KVASIR_V2_DATA)): return KVASIR_V2_DATA
    for dp, dirs, _ in os.walk("/kaggle/input"):
        if any(d.lower().replace(" ", "-").replace("_", "-") == "dyed-lifted-polyps" for d in dirs): return dp
    return KVASIR_V2_DATA
if DATASET == "kvasir_v2":                       # Kvasir-V2: official layout (one folder per class), stratified 70/15/15 split, no augmentation
    PROCESSED = False; DATA_ROOT = _find_kvasir_root(); print("Kvasir-V2 folder used:", DATA_ROOT)
    if not os.path.isdir(DATA_ROOT):
        print("Inputs found under /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "none")
        raise SystemExit("Kvasir-V2 is not attached to this notebook: use Add Input and add the dataset, then run again.")
OUT_DIR        = f"/kaggle/working/out_{DATASET}_{('proc_aug' if USE_AUG_IN_TRAIN else 'proc_noaug') if PROCESSED else 'plain'}{'_QUICK' if QUICK else ''}"
RUN            = ["main", "overlap"]                           # e.g. ["main", "ablation", "baselines", "sweeps", "overlap"]

BACKBONE       = "s"      # "s" -> EfficientNetV2-S, "m" -> EfficientNetV2-M. Pick the one your paper used and
                          # compare the printed parameter count with the manuscript (27.51 M).
CKPT           = None     # eval mode: path to your trained weights, e.g. "/kaggle/input/my-model/best_model.pt"
FINETUNE_EPOCHS = 0       # eval mode: >0 = first fine-tune the loaded model on the train split (validation split picks the best epoch), then test once
FINETUNE_LR    = 3e-5     # head/attention lr during that fine-tuning (backbone gets 0.1x)

SPLIT_SEED     = 42
SEEDS          = [42, 43, 44]
ABL_SEEDS      = [42, 43, 44]                       # 7 variants x 3 seeds is long: run it over several sessions
VAL_FRAC, TEST_FRAC = 0.15, 0.15
IMG_SIZE, BATCH_SIZE = 224, 16
LR, WEIGHT_DECAY = 1e-4, 1e-5
HEAD_EPOCHS, FT_EPOCHS = 5, 35                      # total 40
LABEL_SMOOTHING = 0.05
EMBED_DIM, NUM_HEADS = 512, 8
WINDOW         = 4        # 224 px -> 7x7 map, zero-padded to 8x8 -> four 4x4 windows. WINDOW = 7 would be global attention
DROPOUT        = 0.3
NUM_WORKERS    = 0        # images are cached in RAM, workers are not needed
HASH_NEAR      = 4        # Hamming distance (of 64) counted as a near-duplicate
SWEEP_LRS      = [1e-3, 5e-4, 1e-4, 5e-5, 1e-5]
SWEEP_OPTS     = ["sgd", "rmsprop", "adam"]
if QUICK:
    HEAD_EPOCHS, FT_EPOCHS, SEEDS, ABL_SEEDS = 1, 0, [42], [42]
    print("QUICK MODE: 1 epoch, 1 seed. These numbers are only a pipeline test and must not be used in the paper.")

HYPERKVASIR_13 = {"bbps-0-1", "bbps-2-3", "cecum", "dyed-lifted-polyps", "dyed-resection-margins", "esophagitis",
                  "impacted-stool", "polyps", "pylorus", "retroflex-rectum", "retroflex-stomach",
                  "ulcerative-colitis", "z-line"}
KVASIR_V2_8 = {"dyed-lifted-polyps", "dyed-resection-margins", "esophagitis", "normal-cecum", "normal-pylorus",
               "normal-z-line", "polyps", "ulcerative-colitis"}
EXT = {".jpg", ".jpeg", ".png", ".bmp"}
device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)

# %% DATA ------------------------------------------------------------------
def norm_label(name):
    n = name.lower().strip().replace(" ", "-").replace("_", "-")
    if n.startswith("esophagitis"): return "esophagitis"
    if n.startswith("ulcerative-colitis"): return "ulcerative-colitis"
    return n

def list_images(root, keep):
    rows, seen = [], set()
    for dp, _, files in os.walk(root):
        lab = norm_label(Path(dp).name)
        if lab not in keep: continue
        for f in files:
            if Path(f).suffix.lower() in EXT and (lab, f) not in seen:
                seen.add((lab, f)); rows.append((os.path.join(dp, f), lab))
    return pd.DataFrame(rows, columns=["path", "label"])

def make_split_processed(root):
    """train/ may contain augmented copies named <id>_aug_<hash>.jpg; test/ holds originals.
    Validation is cut from the ORIGINAL training images (by id); augmented copies follow their original into train only."""
    rows = []
    for part in ("train", "test"):
        base = os.path.join(root, part)
        for dp, _, files in os.walk(base):
            if dp == base: continue
            lab = Path(dp).name
            for f in files:
                if Path(f).suffix.lower() in EXT:
                    stem = Path(f).stem
                    rows.append((os.path.join(dp, f), lab, part, "_aug_" in stem, stem.split("_aug_")[0]))
    df = pd.DataFrame(rows, columns=["path", "label", "part", "is_aug", "orig_id"])
    classes = sorted(df.label.unique()); df["y"] = df.label.map({c: i for i, c in enumerate(classes)})
    trn, te = df[df.part == "train"], df[df.part == "test"]
    orig = trn[~trn.is_aug].drop_duplicates("orig_id")
    shared = set(te.orig_id) & set(orig.orig_id)
    print(f"train: {len(orig)} originals + {int(trn.is_aug.sum())} augmented copies | test: {len(te)} | ids shared by train and test: {len(shared)}")
    if "is_aug" in te and te.is_aug.any(): print("WARNING: test folder contains augmented files:", int(te.is_aug.sum()))
    te = te[~te.orig_id.isin(orig.orig_id)]                       # drop any test image that also exists in train
    o_tr, o_va = train_test_split(orig, test_size=PROC_VAL_FRAC, stratify=orig.y, random_state=SPLIT_SEED)
    tr = trn[trn.orig_id.isin(o_tr.orig_id)]
    if not USE_AUG_IN_TRAIN: tr = tr[~tr.is_aug]
    va = trn[trn.orig_id.isin(o_va.orig_id) & ~trn.is_aug]
    print(f"originals per class in train/:\n{orig.label.value_counts().sort_index().to_string()}")
    print(f"augmented copies in training set: {int(tr.is_aug.sum())} | originals in training set: {int((~tr.is_aug).sum())}")
    return classes, df, tr, va, te

def make_split(df):
    classes = sorted(df.label.unique())
    df = df.copy(); df["y"] = df.label.map({c: i for i, c in enumerate(classes)})
    tv, te = train_test_split(df, test_size=TEST_FRAC, stratify=df.y, random_state=SPLIT_SEED)
    tr, va = train_test_split(tv, test_size=VAL_FRAC / (1 - TEST_FRAC), stratify=tv.y, random_state=SPLIT_SEED)
    return classes, tr, va, te

TF = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])      # no augmentation

# Images are decoded and resized ONCE and kept in RAM as uint8 (about 150 KB each). Same preprocessing as before
# (resize to 224 x 224, ImageNet normalisation, no augmentation), but an epoch no longer waits for JPEG decoding.
from concurrent.futures import ThreadPoolExecutor
_CACHE = {}
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def _load(p):
    im = Image.open(p)
    im.draft("RGB", (IMG_SIZE, IMG_SIZE))          # fast JPEG decode at reduced size (no-op for PNG)
    return np.array(im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)

def cache_images(paths):
    todo = [p for p in dict.fromkeys(paths) if p not in _CACHE]
    if not todo: return
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        for p, a in zip(todo, ex.map(_load, todo)): _CACHE[p] = a
    print(f"cached {len(todo)} images in {time.time() - t0:.0f}s")

class DS(Dataset):
    def __init__(self, d):
        self.p, self.y = d.path.tolist(), d.y.tolist(); cache_images(self.p)
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        x = torch.from_numpy(_CACHE[self.p[i]]).permute(2, 0, 1).float().div_(255)
        return (x - _MEAN) / _STD, self.y[i]

def loader(d, shuffle, seed=0):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(DS(d), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=True, generator=g if shuffle else None)

# %% MODELS ----------------------------------------------------------------
class SE(nn.Module):
    def __init__(self, c, r=16):
        super().__init__(); self.fc1, self.fc2 = nn.Linear(c, c // r), nn.Linear(c // r, c)
    def forward(self, x):
        s = torch.sigmoid(self.fc2(torch.relu(self.fc1(x.mean((2, 3))))))
        return x * s[:, :, None, None]

class WindowAttention(nn.Module):
    """Non-overlapping window self-attention; the map is zero-padded to a multiple of the window size."""
    def __init__(self, dim, heads, win):
        super().__init__(); self.win = win
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True); self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        B, C, H, W = x.shape; w = self.win
        ph, pw = (-H) % w, (-W) % w
        if ph or pw: x = nn.functional.pad(x, (0, pw, 0, ph))
        Hp, Wp = H + ph, W + pw
        t = x.permute(0, 2, 3, 1).reshape(B, Hp // w, w, Wp // w, w, C).permute(0, 1, 3, 2, 4, 5).reshape(-1, w * w, C)
        t = self.norm(t + self.attn(t, t, t, need_weights=False)[0])
        t = t.reshape(B, Hp // w, Wp // w, w, w, C).permute(0, 1, 3, 2, 4, 5).reshape(B, Hp, Wp, C)
        return t.permute(0, 3, 1, 2)[:, :, :H, :W]

class AxialAttentionH(nn.Module):
    """Attention along the height axis only."""
    def __init__(self, dim, heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True); self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        B, C, H, W = x.shape
        t = x.permute(0, 3, 2, 1).reshape(B * W, H, C)
        t = self.norm(t + self.attn(t, t, t, need_weights=False)[0])
        return t.reshape(B, W, H, C).permute(0, 3, 2, 1)

def effnet_features():
    if BACKBONE == "m":
        return models.efficientnet_v2_m(weights=models.EfficientNet_V2_M_Weights.DEFAULT).features
    return models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT).features

class HEASNet(nn.Module):
    """Variants: se / win / ax switches. With all three off this is the EfficientNetV2 base model."""
    def __init__(self, n_classes, se=True, win=True, ax=True):
        super().__init__()
        self.backbone = effnet_features()
        self.proj = nn.Conv2d(1280, EMBED_DIM, 1)
        self.se = SE(EMBED_DIM) if se else nn.Identity()
        self.win = WindowAttention(EMBED_DIM, NUM_HEADS, WINDOW) if win else nn.Identity()
        self.ax = AxialAttentionH(EMBED_DIM, NUM_HEADS) if ax else nn.Identity()
        self.drop, self.fc = nn.Dropout(DROPOUT), nn.Linear(EMBED_DIM, n_classes)
    def attn_modules(self): return [self.proj, self.se, self.win, self.ax]
    def forward(self, x):
        x = self.ax(self.win(self.se(self.proj(self.backbone(x)))))
        return self.fc(self.drop(x.mean((2, 3))))

VARIANTS = {"base": (False, False, False), "+SE": (True, False, False), "+Swin": (False, True, False),
            "+Swin+SE": (True, True, False), "+Axial": (False, False, True), "+SE+Axial": (True, False, True),
            "HEASNet": (True, True, True)}

class Baseline(nn.Module):
    def __init__(self, name, n_classes):
        super().__init__(); self.name = name
        if name == "ResNet50":
            m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT); m.fc = nn.Linear(2048, n_classes)
        elif name == "EfficientNetV2":
            m = (models.efficientnet_v2_m(weights=models.EfficientNet_V2_M_Weights.DEFAULT) if BACKBONE == "m"
                 else models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT))
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
        elif name == "ConvNeXt-T":
            m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
            m.classifier[2] = nn.Linear(m.classifier[2].in_features, n_classes)
        elif name == "ViT-B/16":
            m = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
            m.heads.head = nn.Linear(m.heads.head.in_features, n_classes)
        else: raise ValueError(name)
        self.m = m
    def forward(self, x): return self.m(x)
    def attn_modules(self): return []

# %% METRICS (pure numpy: no torch needed) ---------------------------------
def ece_score(probs, y, bins=15):
    conf, pred = probs.max(1), probs.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def mcnemar(y, pred_a, pred_b):
    a_ok, b_ok = pred_a == y, pred_b == y
    b, c = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())      # b: only A right, c: only B right
    n = b + c
    if n == 0: return dict(only_a=b, only_b=c, p_value=1.0, test="none")
    if n < 25: return dict(only_a=b, only_b=c, p_value=float(binomtest(b, n, 0.5).pvalue), test="exact binomial")
    stat = (abs(b - c) - 1) ** 2 / n
    return dict(only_a=b, only_b=c, p_value=float(chi2.sf(stat, 1)), test="chi-square (continuity corrected)")

def boot_ci(y, pred, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    v = [accuracy_score(y[i], pred[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def full_metrics(y, probs):
    K = probs.shape[1]; pred = probs.argmax(1)
    p, r, f1, sup = precision_recall_fscore_support(y, pred, labels=range(K), zero_division=0)
    mp, mr, mf, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    wf = precision_recall_fscore_support(y, pred, average="weighted", zero_division=0)[2]
    oh = np.eye(K)[y]
    return dict(acc=accuracy_score(y, pred), acc_ci=boot_ci(y, pred), macro_precision=mp, macro_recall=mr, macro_f1=mf,
                weighted_f1=wf, kappa=cohen_kappa_score(y, pred), mcc=matthews_corrcoef(y, pred),
                roc_auc_macro=roc_auc_score(y, probs, multi_class="ovr", average="macro", labels=list(range(K))),
                ece=ece_score(probs, y)), (p, r, f1, sup, [roc_auc_score(oh[:, k], probs[:, k]) for k in range(K)])

# %% TRAINING --------------------------------------------------------------
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def run_epoch(model, dl, crit, opt=None, scaler=None):
    train = opt is not None; model.train(train); tl = co = n = 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                out = model(x); loss = crit(out, y)
            if train:
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(y); co += (out.argmax(1) == y).sum().item(); n += len(y)
    return tl / n, co / n

def make_opt(name, groups, lr):
    ps = [dict(params=g["params"], lr=g["lr"]) for g in groups]
    if name == "adam": return torch.optim.Adam(ps, weight_decay=WEIGHT_DECAY)
    if name == "sgd": return torch.optim.SGD(ps, momentum=0.9, weight_decay=WEIGHT_DECAY)
    if name == "rmsprop": return torch.optim.RMSprop(ps, weight_decay=WEIGHT_DECAY)
    raise ValueError(name)

def train_model(model, dl_tr, dl_va, lr=LR, opt_name="adam", epochs=(HEAD_EPOCHS, FT_EPOCHS)):
    """Two-stage training for HEASNet variants; baselines are fine-tuned end-to-end for the same number of epochs."""
    model.to(device); crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    best, state, hist = -1.0, None, []
    def stage(groups, n_ep, tag):
        nonlocal best, state
        opt = make_opt(opt_name, groups, lr)
        for ep in range(n_ep):
            t_ep = time.time()
            trl, tra = run_epoch(model, dl_tr, crit, opt, scaler); vl, va = run_epoch(model, dl_va, crit)
            print(f"  stage {tag} epoch {ep + 1}/{n_ep}: train {tra:.4f} val {va:.4f} ({time.time() - t_ep:.0f}s)")
            hist.append(dict(stage=tag, epoch=len(hist) + 1, train_loss=trl, train_acc=tra, val_loss=vl, val_acc=va))
            if va > best: best, state = va, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if isinstance(model, HEASNet):
        for p in model.parameters(): p.requires_grad = False
        head = [p for m in model.attn_modules() for p in m.parameters()] + list(model.fc.parameters())
        for p in head: p.requires_grad = True
        stage([dict(params=head, lr=lr)], epochs[0], 1)                       # stage 1: new layers only
        for p in model.parameters(): p.requires_grad = True
        bb = list(model.backbone.parameters()); ids = {id(p) for p in bb}
        stage([dict(params=bb, lr=lr * 0.1), dict(params=[p for p in model.parameters() if id(p) not in ids], lr=lr)], epochs[1], 2)
    else:
        stage([dict(params=list(model.parameters()), lr=lr * 0.1)], sum(epochs), 2)
    model.load_state_dict(state)
    return best, pd.DataFrame(hist)

@torch.no_grad()
def predict(model, dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            o = model(x.to(device))
        P.append(torch.softmax(o.float(), 1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

def efficiency(model):
    out = dict(params_M=sum(p.numel() for p in model.parameters()) / 1e6, gmacs=None, latency_ms=None, peak_mem_MB=None)
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device); model.eval()
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as fc, torch.no_grad(): model(x)
        out["gmacs"] = fc.get_total_flops() / 2 / 1e9
    except Exception as e: print("GMACs not computed:", e)
    if device == "cuda":
        with torch.no_grad():
            for _ in range(10): model(x)
            torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.time()
            for _ in range(100): model(x)
            torch.cuda.synchronize()
        out["latency_ms"] = (time.time() - t0) / 100 * 1000; out["peak_mem_MB"] = torch.cuda.max_memory_allocated() / 2**20
    return out

def one_run(name, builder, seed, dls, **kw):
    """Train, pick best-val checkpoint, evaluate val and test, save predictions."""
    f = f"{OUT_DIR}/preds_{name.replace('/', '_')}_s{seed}.npz"
    if os.path.exists(f):
        z = np.load(f); return dict(val_acc=float(z["val_acc"]), probs=z["probs"], y=z["y"], cached=True)
    seed_all(seed); model = builder(); dl_tr, dl_va, dl_te = dls
    best, hist = train_model(model, dl_tr, dl_va, **kw)
    probs, y = predict(model, dl_te)
    np.savez(f, probs=probs, y=y, val_acc=best); hist.to_csv(f.replace("preds_", "history_").replace(".npz", ".csv"), index=False)
    if name == "HEASNet":      # best-validation weights, accuracies in the file name
        torch.save(model.state_dict(), f.replace("preds_", "model_").replace(".npz", f"_val{100 * best:.2f}_test{100 * accuracy_score(y, probs.argmax(1)):.2f}.pt"))
    res = dict(val_acc=best, probs=probs, y=y, cached=False, model=model)
    print(f"[{name} seed {seed}] val {best:.4f} test {accuracy_score(y, probs.argmax(1)):.4f}")
    return res

# %% SETUP ----------------------------------------------------------------
if PROCESSED:
    classes, df, tr, va, te = make_split_processed(DATA_ROOT)
else:
    df = list_images(DATA_ROOT, HYPERKVASIR_13 if DATASET == "hyperkvasir" else KVASIR_V2_8)
    classes, tr, va, te = make_split(df)
K = len(classes)
print("device:", device)
print(f"{DATASET}: {len(df)} images, {K} classes | train {len(tr)} val {len(va)} test {len(te)} | backbone EfficientNetV2-{BACKBONE.upper()}")
if not PROCESSED:
    print("expected totals: hyperkvasir 10,553; kvasir_v2 8,000 (8 classes x 1,000)")
    print(df.label.value_counts().to_string())
    if DATASET == "kvasir_v2": assert len(df) == 8000 and df.label.nunique() == 8, "Kvasir-V2 not read correctly: paste the folder listing back to me"
pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")]).to_csv(f"{OUT_DIR}/splits.csv", index=False)
pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")]).pivot_table(
    index="label", columns="split", values="y", aggfunc="count").to_csv(f"{OUT_DIR}/class_split_counts.csv")
dls = (loader(tr, True), loader(va, False), loader(te, False))

# %% FULL REPORT FOR ONE MODEL: metrics, confusion matrices, ROC, efficiency, statistics, confidence, Grad-CAM ----
def _wilson(k, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def _plot_cm(cm, names, fn, title):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7.6, 6), dpi=200); im = ax.imshow(cm, cmap="Blues"); k = len(names)
    ax.set_xticks(range(k)); ax.set_yticks(range(k)); ax.set_xticklabels(names, rotation=60, ha="right", fontsize=7); ax.set_yticklabels(names, fontsize=7)
    for i in range(k):
        for j in range(k):
            if cm[i, j]: ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontsize=9); fig.colorbar(im, fraction=.046)
    fig.tight_layout(); fig.savefig(fn); plt.close(fig)

def efficiency_full(model):
    import copy
    out = dict(precision="FP32", gpu=(torch.cuda.get_device_name(0) if device == "cuda" else "cpu"))
    out["params_M"] = sum(p.numel() for p in model.parameters()) / 1e6
    out["weights_MB_fp32"] = sum(p.numel() for p in model.parameters()) * 4 / 2**20
    model.eval(); x1 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as fc, torch.no_grad(): model(x1)
        out["gmacs"] = fc.get_total_flops() / 2 / 1e9; out["gflops"] = fc.get_total_flops() / 1e9
    except Exception as e: print("FLOP counter failed:", e)
    if device == "cuda":
        with torch.no_grad():
            for _ in range(20): model(x1)
            torch.cuda.reset_peak_memory_stats(); ts = []
            for _ in range(200):
                torch.cuda.synchronize(); t0 = time.perf_counter(); model(x1); torch.cuda.synchronize(); ts.append((time.perf_counter() - t0) * 1000)
            out["gpu_latency_ms_mean"] = float(np.mean(ts)); out["gpu_latency_ms_sd"] = float(np.std(ts)); out["gpu_latency_ms_p95"] = float(np.percentile(ts, 95))
            out["gpu_peak_mem_MB_batch1"] = torch.cuda.max_memory_allocated() / 2**20
            xb = torch.randn(16, 3, IMG_SIZE, IMG_SIZE, device=device)
            for _ in range(5): model(xb)
            torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.perf_counter()
            for _ in range(30): model(xb)
            torch.cuda.synchronize(); dt = time.perf_counter() - t0
            out["gpu_throughput_img_per_s_batch16"] = 30 * 16 / dt; out["gpu_peak_mem_MB_batch16"] = torch.cuda.max_memory_allocated() / 2**20
    try:
        cm_ = copy.deepcopy(model).float().cpu().eval(); xc = torch.randn(1, 3, IMG_SIZE, IMG_SIZE); ts = []
        with torch.no_grad():
            for _ in range(3): cm_(xc)
            for _ in range(15):
                t0 = time.perf_counter(); cm_(xc); ts.append((time.perf_counter() - t0) * 1000)
        out["cpu_latency_ms_mean"] = float(np.mean(ts)); out["cpu_latency_ms_sd"] = float(np.std(ts)); del cm_
    except Exception as e: print("CPU latency skipped:", e)
    return out

def gradcam_report(model, dset, probs, y, outdir, per_class=2, n_err=12):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    import torch.nn.functional as F
    model.eval(); pred = probs.argmax(1); conf = probs.max(1); store = {}
    def hook(name):
        def f(m, i, o): store[name] = o
        return f
    hs = [getattr(model, n).register_forward_hook(hook(n)) for n in ("se", "win", "ax")]
    def maps(i):
        x, _ = dset[i]; x = x.unsqueeze(0).to(device); c = int(pred[i])
        with torch.enable_grad():
            store.clear(); out = model(x.float()); feat = store["ax"]
            g = torch.autograd.grad(out[0, c], feat)[0]
        cam = F.relu((g.mean((2, 3), keepdim=True) * feat).sum(1, keepdim=True)).detach()
        up = lambda t: F.interpolate(t, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
        nz = lambda a: (a - a.min()) / (a.max() - a.min() + 1e-8)
        br = {n: nz(up(store[n].detach().abs().mean(1, keepdim=True))) for n in ("se", "win", "ax")}
        img = (x[0].cpu() * _STD + _MEAN).clamp(0, 1).permute(1, 2, 0).numpy()
        return img, nz(up(cam)), br
    rng = np.random.RandomState(0); K_ = probs.shape[1]; pick = []
    for c in range(K_):
        ok = np.where((y == c) & (pred == c))[0]
        if len(ok): pick += [(int(i), "correct") for i in rng.choice(ok, min(per_class, len(ok)), replace=False)]
    bad = np.where(pred != y)[0]; bad = bad[np.argsort(-conf[bad])][:n_err]; pick_err = [(int(i), "error") for i in bad]
    corr = {n: [[] for _ in range(K_)] for n in ("se", "win", "ax")}
    def draw(items, fn, cols=4):
        rows = int(math.ceil(len(items) / cols)); fig, axs = plt.subplots(rows, cols * 2, figsize=(cols * 3.2, rows * 1.7), dpi=150); axs = np.atleast_2d(axs)
        for a in axs.ravel(): a.axis("off")
        for k, (i, kind) in enumerate(items):
            img, cam, br = maps(i); r, c0 = divmod(k, cols)
            for n in br:
                cc = np.corrcoef(cam.ravel(), br[n].ravel())[0, 1]; corr[n][int(pred[i])].append(0.0 if np.isnan(cc) else cc)
            axs[r, c0 * 2].imshow(img); axs[r, c0 * 2 + 1].imshow(img); axs[r, c0 * 2 + 1].imshow(cam, cmap="jet", alpha=.45)
            axs[r, c0 * 2].set_title(f"T:{classes[y[i]][:12]}\nP:{classes[pred[i]][:12]} {conf[i]:.2f}", fontsize=5.5, color=("green" if kind == "correct" else "red"))
        fig.tight_layout(); fig.savefig(fn); plt.close(fig)
    draw(pick, f"{outdir}/gradcam_correct_per_class.png"); draw(pick_err, f"{outdir}/gradcam_misclassified.png")
    for h in hs: h.remove()
    rows = [dict(**{"class": classes[c]}, **{f"corr_{n}": (float(np.mean(corr[n][c])) if corr[n][c] else float("nan")) for n in corr}, n_samples=len(corr["se"][c])) for c in range(K_)]
    pd.DataFrame(rows).to_csv(f"{outdir}/gradcam_attention_correlation.csv", index=False)

def full_report(model, outdir, dls_, MAIN_OUT_DIR=None):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve, auc, log_loss
    os.makedirs(outdir, exist_ok=True); names = [c.capitalize() for c in classes]; Kc = len(classes)
    sets = {"train": loader(tr, False), "val": dls_[1], "test": dls_[2]}; res = {}
    t0 = time.perf_counter()
    for s, dl in sets.items():
        P, Y = predict(model, dl); m, pc = full_metrics(Y, P); res[s] = (P, Y, m, pc)
    print(f"prediction time for {len(tr) + len(va) + len(te)} images: {time.perf_counter() - t0:.1f}s")
    rows = []
    for s, (P, Y, m, pc) in res.items():
        cm = confusion_matrix(Y, P.argmax(1), labels=range(Kc))
        pd.DataFrame(cm, index=classes, columns=classes).to_csv(f"{outdir}/confusion_matrix_{s}.csv"); _plot_cm(cm, names, f"{outdir}/fig_confusion_{s}.png", f"{s} set (n={len(Y)})")
        spec = [(cm.sum() - cm[k].sum() - cm[:, k].sum() + cm[k, k]) / max(cm.sum() - cm[k].sum(), 1) for k in range(Kc)]
        pd.DataFrame(dict(**{"class": classes}, precision=pc[0], recall_sensitivity=pc[1], specificity=spec, f1=pc[2], support=pc[3], roc_auc=pc[4])).to_csv(f"{outdir}/classification_report_{s}.csv", index=False)
        rows.append(dict(split=s, n=len(Y), **{k: v for k, v in m.items() if k != "acc_ci"}, acc_ci_lo=m["acc_ci"][0], acc_ci_hi=m["acc_ci"][1],
                         brier=float(((P - np.eye(Kc)[Y]) ** 2).sum(1).mean()), nll=float(log_loss(Y, P, labels=list(range(Kc))))))
    pd.DataFrame(rows).to_csv(f"{outdir}/split_metrics.csv", index=False)
    P, Y, m, pc = res["test"]; pred = P.argmax(1); conf = P.max(1); ok = pred == Y
    fig, ax = plt.subplots(figsize=(6.6, 5.2), dpi=200); oh = np.eye(Kc)[Y]
    for c in range(Kc):
        fpr, tpr, _ = roc_curve(oh[:, c], P[:, c]); ax.plot(fpr, tpr, lw=1.1, label=f"{names[c]} ({auc(fpr, tpr):.3f})")
    fpr, tpr, _ = roc_curve(oh.ravel(), P.ravel()); ax.plot(fpr, tpr, "k--", lw=1.5, label=f"micro ({auc(fpr, tpr):.3f})"); ax.plot([0, 1], [0, 1], "k:", lw=.8)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend(fontsize=6, loc="lower right"); fig.tight_layout(); fig.savefig(f"{outdir}/fig_roc_test.png"); plt.close(fig)
    rng = np.random.RandomState(0); boots = []
    for _ in range(500):
        i = rng.randint(0, len(Y), len(Y)); yb, pb = Y[i], P[i]
        try: auc_b = roc_auc_score(yb, pb, multi_class="ovr", average="macro", labels=list(range(Kc)))
        except Exception: auc_b = np.nan
        pr = pb.argmax(1); boots.append([accuracy_score(yb, pr), precision_recall_fscore_support(yb, pr, average="macro", zero_division=0)[2], cohen_kappa_score(yb, pr), matthews_corrcoef(yb, pr), auc_b])
    boots = np.array(boots); est = [m["acc"], m["macro_f1"], m["kappa"], m["mcc"], m["roc_auc_macro"]]
    pd.DataFrame([dict(metric=n, estimate=e, ci_lo=np.nanpercentile(boots[:, k], 2.5), ci_hi=np.nanpercentile(boots[:, k], 97.5), bootstrap_sd=np.nanstd(boots[:, k]))
                  for k, (n, e) in enumerate(zip(["accuracy", "macro_f1", "kappa", "mcc", "macro_roc_auc"], est))]).to_csv(f"{outdir}/bootstrap_ci_test.csv", index=False)
    maj = np.bincount(Y).max() / len(Y); k_ok = int(ok.sum())
    stat = dict(n_test=len(Y), correct=k_ok, majority_class_rate=float(maj), binom_p_vs_majority=float(binomtest(k_ok, len(Y), maj, alternative="greater").pvalue),
                binom_p_vs_chance=float(binomtest(k_ok, len(Y), 1 / Kc, alternative="greater").pvalue), kappa=m["kappa"], mcc=m["mcc"])
    pd.DataFrame([dict(**{"class": classes[k]}, recall=pc[1][k], ci_lo=_wilson(int(round(pc[1][k] * pc[3][k])), int(pc[3][k]))[0], ci_hi=_wilson(int(round(pc[1][k] * pc[3][k])), int(pc[3][k]))[1], support=int(pc[3][k])) for k in range(Kc)]).to_csv(f"{outdir}/per_class_recall_ci_test.csv", index=False)
    import glob; mc = []
    for d_ in {x for x in (OUT_DIR, MAIN_OUT_DIR) if x}:
        for f_ in glob.glob(f"{d_}/preds_*_s{SEEDS[0]}.npz"):
            nm = os.path.basename(f_)[6:-4]
            if d_ == outdir or nm.startswith("HEASNet"): continue
            z = np.load(f_)
            if len(z["y"]) == len(Y) and (z["y"] == Y).all():
                r_ = mcnemar(Y, pred, z["probs"].argmax(1)); r_["compared_with"] = nm; r_["acc_other"] = float((z["probs"].argmax(1) == Y).mean()); mc.append(r_)
    if mc: pd.DataFrame(mc).to_csv(f"{outdir}/mcnemar_vs_other_models.csv", index=False)
    json.dump(stat, open(f"{outdir}/statistical_tests.json", "w"), indent=2, default=float)
    ent = -(P * np.log(P + 1e-12)).sum(1); edges = np.linspace(0, 1, 16); gaps, rel = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        b = (conf > lo) & (conf <= hi)
        if b.any(): gaps.append(abs(ok[b].mean() - conf[b].mean())); rel.append((conf[b].mean(), ok[b].mean(), int(b.sum())))
    conf_sum = dict(mean_conf_all=float(conf.mean()), mean_conf_correct=float(conf[ok].mean()), mean_conf_wrong=float(conf[~ok].mean()) if (~ok).any() else None,
                    mean_entropy_correct=float(ent[ok].mean()), mean_entropy_wrong=float(ent[~ok].mean()) if (~ok).any() else None, ece_15bins=m["ece"],
                    mce_15bins=float(max(gaps)), brier=rows[2]["brier"], nll=rows[2]["nll"])
    json.dump(conf_sum, open(f"{outdir}/confidence_summary.json", "w"), indent=2)
    fig, ax = plt.subplots(figsize=(4.6, 4.4), dpi=200); ax.plot([0, 1], [0, 1], "k:"); ax.plot([r[0] for r in rel], [r[1] for r in rel], "o-", color="#1f6f8b")
    ax.set_xlabel("Mean confidence"); ax.set_ylabel("Accuracy"); ax.set_title(f"Reliability diagram (ECE {m['ece']:.3f})", fontsize=9); fig.tight_layout(); fig.savefig(f"{outdir}/fig_reliability.png"); plt.close(fig)
    pd.DataFrame([dict(conf_threshold=t, coverage=float((conf >= t).mean()), accuracy_on_covered=float(ok[conf >= t].mean()) if (conf >= t).any() else float("nan"))
                  for t in (0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99)]).to_csv(f"{outdir}/selective_accuracy_test.csv", index=False)
    eff = efficiency_full(model); json.dump(eff, open(f"{outdir}/efficiency_full.json", "w"), indent=2, default=float); print(json.dumps(eff, indent=2, default=float))
    try: gradcam_report(model, dls_[2].dataset, P, Y, outdir)
    except Exception as e: print("Grad-CAM skipped:", repr(e))
    print("test accuracy", round(m["acc"], 4), "| kappa", round(m["kappa"], 4), "| mcc", round(m["mcc"], 4), "| macro AUC", round(m["roc_auc_macro"], 4))
    import shutil; z = shutil.make_archive(outdir.rstrip("/") + "_download", "zip", outdir); print("ZIP with all tables and figures (download this):", z)

# %% EVAL ONLY: use a model you already trained ------------------------------------
if "eval" in RUN:
    assert CKPT and os.path.exists(CKPT), "set CKPT to the path of your trained weights"
    MAIN_OUT_DIR = OUT_DIR; OUT_DIR = OUT_DIR + "_eval"; os.makedirs(OUT_DIR, exist_ok=True)
    try:
        ck = torch.load(CKPT, map_location="cpu")
    except Exception:
        ck = torch.load(CKPT, map_location="cpu", weights_only=False)   # whole-model pickle (your own file)
    if isinstance(ck, nn.Module): model = ck.to(device)
    else:
        if isinstance(ck, dict):
            for k in ("state_dict", "model_state_dict", "model", "net"):
                if k in ck and isinstance(ck[k], dict): ck = ck[k]; break
        ck = {k.replace("module.", ""): v for k, v in ck.items()}
        model, found = None, None
        print(f"checkpoint: {len(ck)} tensors; first keys:")
        for k_, v_ in list(ck.items())[:6]: print("   ", k_, tuple(v_.shape))
        print("    ...  last keys:")
        for k_, v_ in list(ck.items())[-4:]: print("   ", k_, tuple(v_.shape))
        fcw = [v_ for k_, v_ in ck.items() if k_.endswith("fc.weight") or k_.endswith("classifier.weight")]
        if fcw: print(f"classes in checkpoint: {fcw[-1].shape[0]} | classes in current data: {K}")
        errs = []
        for bb in dict.fromkeys([BACKBONE, "s", "m"]):          # try the settings the checkpoint may have been trained with
            for wn in dict.fromkeys([CKPT_WINDOW, 7, 4]):
                BACKBONE, WINDOW = bb, wn
                try:
                    cand = HEASNet(K); res = cand.load_state_dict(ck, strict=False)
                    if not res.missing_keys and not res.unexpected_keys: model, found = cand.to(device), (bb, wn); break
                    errs.append((bb, wn, "missing", res.missing_keys[:3], "unexpected", res.unexpected_keys[:3]))
                except Exception as e: errs.append((bb, wn, str(e)[:300]))
            if model is not None: break
        if model is None:
            for e_ in errs[:3]: print("tried:", e_)
            raise SystemExit("The checkpoint does not fit this model code/class set. Paste the 'checkpoint:' and 'tried:' lines above back to me.")
        print(f"checkpoint matched: backbone EfficientNetV2-{found[0].upper()}, window {found[1]}")
    if FINETUNE_EPOCHS > 0:
        print(f"fine-tuning the loaded model for {FINETUNE_EPOCHS} epochs on the train split (validation split selects the best epoch)")
        _t = time.time(); _b, _h = train_model(model, dls[0], dls[1], lr=FINETUNE_LR, epochs=(0, FINETUNE_EPOCHS))
        _h.to_csv(f"{OUT_DIR}/history_finetune.csv", index=False)
        print(f"fine-tuning done: best val {_b:.4f}, {(time.time() - _t) / FINETUNE_EPOCHS:.0f}s per epoch")
    print("!! The result is valid only if this model was NOT trained on any image of the test folder. "
          "A model from the first run (which pooled train and test) must not be evaluated on the test folder.")
    probs, y = predict(model, dls[2]); m, pc = full_metrics(y, probs)
    pv, yv = predict(model, dls[1]); val_acc = accuracy_score(yv, pv.argmax(1))
    ptr, ytr = predict(model, loader(tr, False)); print(f"accuracy of this checkpoint: train {accuracy_score(ytr, ptr.argmax(1)):.4f} | val {val_acc:.4f} | test {m['acc']:.4f}")
    np.savez(f"{OUT_DIR}/preds_HEASNet_s{SEEDS[0]}.npz", probs=probs, y=y, val_acc=val_acc)
    pd.DataFrame(dict(**{"class": classes}, precision=pc[0], recall=pc[1], f1=pc[2], support=pc[3], roc_auc=pc[4])).to_csv(
        f"{OUT_DIR}/test_classification_report.csv", index=False)
    pd.DataFrame(confusion_matrix(y, probs.argmax(1)), index=classes, columns=classes).to_csv(f"{OUT_DIR}/confusion_matrix.csv")
    eff = efficiency(model)
    summary = dict(mode="evaluation of a supplied checkpoint (no training)", checkpoint=CKPT, dataset=DATASET, n_classes=K,
                   n_train=len(tr), n_val=len(va), n_test=len(te), val_acc=val_acc,
                   test=dict(**{k: v for k, v in m.items() if k != "acc_ci"}, acc_ci_lo=m["acc_ci"][0], acc_ci_hi=m["acc_ci"][1]), efficiency=eff)
    json.dump(summary, open(f"{OUT_DIR}/test_metrics.json", "w"), indent=2, default=float); print(json.dumps(summary, indent=2, default=float))
    _tag = f"val{100 * val_acc:.2f}_test{100 * m['acc']:.2f}"
    torch.save(model.state_dict(), f"{OUT_DIR}/best_model_{'finetuned_' if FINETUNE_EPOCHS > 0 else ''}{_tag}.pth")   # the model that was evaluated, accuracies in the name
    print("saved:", f"{OUT_DIR}/best_model_{'finetuned_' if FINETUNE_EPOCHS > 0 else ''}{_tag}.pth")
    full_report(model, OUT_DIR, dls, MAIN_OUT_DIR)

# %% MAIN -------------------------------------------------------------------
if "main" in RUN:
    accs, rows = [], []
    for s in SEEDS:
        r = one_run("HEASNet", lambda: HEASNet(K), s, dls); m, pc = full_metrics(r["y"], r["probs"]); accs.append(m["acc"])
        rows.append(dict(seed=s, val_acc=r["val_acc"], **{k: v for k, v in m.items() if k != "acc_ci"}, acc_ci_lo=m["acc_ci"][0], acc_ci_hi=m["acc_ci"][1]))
        if s == SEEDS[0]:
            pd.DataFrame(dict(**{"class": classes}, precision=pc[0], recall=pc[1], f1=pc[2], support=pc[3], roc_auc=pc[4])).to_csv(
                f"{OUT_DIR}/test_classification_report.csv", index=False)
            pd.DataFrame(confusion_matrix(r["y"], r["probs"].argmax(1)), index=classes, columns=classes).to_csv(f"{OUT_DIR}/confusion_matrix.csv")
            eff = efficiency(r["model"] if "model" in r else HEASNet(K).to(device))
    per_seed = pd.DataFrame(rows); per_seed.to_csv(f"{OUT_DIR}/main_per_seed.csv", index=False)
    summary = dict(dataset=DATASET, n_classes=K, n_train=len(tr), n_val=len(va), n_test=len(te), backbone=f"EfficientNetV2-{BACKBONE.upper()}",
                   seeds=SEEDS, test_accuracy_mean=float(np.mean(accs)), test_accuracy_sd=float(np.std(accs, ddof=1)) if len(accs) > 1 else None,
                   first_seed=rows[0], efficiency=eff, augmentation=("offline copies in training only" if (PROCESSED and USE_AUG_IN_TRAIN) else "none"), epochs_total=HEAD_EPOCHS + FT_EPOCHS)
    json.dump(summary, open(f"{OUT_DIR}/test_metrics.json", "w"), indent=2, default=float)
    print(json.dumps(summary, indent=2, default=float))

# %% ABLATION ----------------------------------------------------------------
if "ablation" in RUN:
    rows = []
    for name, (se, win, ax) in VARIANTS.items():
        accs = []
        for s in ABL_SEEDS:
            r = one_run("HEASNet" if name == "HEASNet" else f"abl_{name}", lambda: HEASNet(K, se, win, ax), s, dls); accs.append(accuracy_score(r["y"], r["probs"].argmax(1)))
        e = efficiency(HEASNet(K, se, win, ax).to(device))
        rows.append(dict(variant=name, EfficientNetV2="Yes", Swin=("Yes" if win else "No"), Axial=("Yes" if ax else "No"), SE=("Yes" if se else "No"),
                         params_M=e["params_M"], gmacs=e["gmacs"], acc_mean=100 * np.mean(accs), acc_sd=100 * (np.std(accs, ddof=1) if len(accs) > 1 else float("nan")), n_seeds=len(accs)))
        pd.DataFrame(rows).to_csv(f"{OUT_DIR}/ablation_results.csv", index=False)
    print(pd.DataFrame(rows).round(3).to_string())

# %% BASELINES + McNEMAR --------------------------------------------------------
if "baselines" in RUN:
    rows, best_base, best_acc = [], None, -1
    for name in ["ResNet50", "EfficientNetV2", "ConvNeXt-T", "ViT-B/16"]:
        accs = []
        for s in SEEDS:
            r = one_run(f"base_{name}", lambda: Baseline(name, K), s, dls); accs.append(accuracy_score(r["y"], r["probs"].argmax(1)))
            if s == SEEDS[0] and np.mean(accs) > best_acc: best_acc, best_base = np.mean(accs), name
        e = efficiency(Baseline(name, K).to(device))
        rows.append(dict(model=name, acc_mean=100 * np.mean(accs), acc_sd=100 * (np.std(accs, ddof=1) if len(accs) > 1 else float("nan")), **e))
    pd.DataFrame(rows).to_csv(f"{OUT_DIR}/baselines.csv", index=False); print(pd.DataFrame(rows).round(3).to_string())
    a = np.load(f"{OUT_DIR}/preds_HEASNet_s{SEEDS[0]}.npz"); b = np.load(f"{OUT_DIR}/preds_base_{best_base}_s{SEEDS[0]}.npz")
    res = mcnemar(a["y"], a["probs"].argmax(1), b["probs"].argmax(1)); res["strongest_baseline"] = best_base
    pd.DataFrame([res]).to_csv(f"{OUT_DIR}/mcnemar.csv", index=False); print(res)

# %% SWEEPS (validation accuracy only; Tables 9 and 10) --------------------------
if "sweeps" in RUN:
    rows = []
    for lr in SWEEP_LRS:
        r = one_run("HEASNet" if lr == LR else f"sweep_adam_lr{lr:g}", lambda: HEASNet(K), SEEDS[0], dls, lr=lr)
        rows.append(dict(kind="learning_rate", setting=f"{lr:g}", val_acc=100 * r["val_acc"]))
    for o in SWEEP_OPTS:
        if o == "adam": rows.append(dict(kind="optimizer", setting="adam", val_acc=[x for x in rows if x["setting"] == f"{LR:g}"][0]["val_acc"])); continue
        r = one_run(f"sweep_{o}", lambda: HEASNet(K), SEEDS[0], dls, opt_name=o)
        rows.append(dict(kind="optimizer", setting=o, val_acc=100 * r["val_acc"]))
    pd.DataFrame(rows).to_csv(f"{OUT_DIR}/sweeps.csv", index=False); print(pd.DataFrame(rows).round(2).to_string())
    print("NOTE: Tables 9-10 report validation accuracy only; the 1e-4 row and the Adam row are the main seed-42 run.")

# %% OVERLAP / NEAR-DUPLICATE CHECKS --------------------------------------------
_POP = np.array([bin(i).count("1") for i in range(65536)], dtype=np.uint8)
def popcount64(x): return _POP[x.view(np.uint16).reshape(-1, 4)].sum(1)

def dhash(path, size=8):
    im = Image.open(path); im.draft("L", (4 * size, 4 * size))
    im = im.convert("L").resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(im, dtype=np.int16); bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

def min_dist(queries, ref):
    q, r = np.array(queries, dtype=np.uint64), np.array(ref, dtype=np.uint64)
    return np.array([popcount64(r ^ h).min() for h in q])

if "overlap" in RUN:
    h = {s: [dhash(p) for p in part.path] for s, part in (("train", tr), ("val", va), ("test", te))}
    out = {"dataset": DATASET, "hash": "dHash 64-bit", "near_threshold_bits": HASH_NEAR}
    for s in ("val", "test"):
        d_ = min_dist(h[s], h["train"])
        out[f"{s}_exact_duplicates_in_train"] = int((d_ == 0).sum()); out[f"{s}_near_duplicates_in_train"] = int((d_ <= HASH_NEAR).sum()); out[f"{s}_n"] = len(h[s])
    hk_root = HYPERKVASIR_ROOT if os.path.isdir(HYPERKVASIR_ROOT) else HK_PROCESSED
    if DATASET == "kvasir_v2" and os.path.isdir(hk_root):
        if hk_root == HK_PROCESSED:                  # processed copy: originals only (skip *_aug_* files)
            hk = pd.DataFrame([(os.path.join(dp, f),) for dp, _, fs in os.walk(hk_root) for f in fs if Path(f).suffix.lower() in EXT and "_aug_" not in f], columns=["path"])
        else: hk = list_images(hk_root, HYPERKVASIR_13)
        hh = [dhash(p) for p in hk.path]; out["hyperkvasir_reference_images"] = len(hh)
        allv = [dhash(p) for p in df.path]; d_ = min_dist(allv, hh)
        out["kvasir_v2_images"] = len(allv); out["kvasir_v2_exact_in_hyperkvasir"] = int((d_ == 0).sum()); out["kvasir_v2_near_in_hyperkvasir"] = int((d_ <= HASH_NEAR).sum())
    json.dump(out, open(f"{OUT_DIR}/overlap_check.json", "w"), indent=2); print(json.dumps(out, indent=2))
    # accuracy of the main model on test images WITHOUT a near-duplicate in the training set
    dt = min_dist(h["test"], h["train"]); keep = dt > HASH_NEAR; rows_dd = []
    for sd in SEEDS:
        f_ = f"{OUT_DIR}/preds_HEASNet_s{sd}.npz"
        if os.path.exists(f_):
            z = np.load(f_); ok = z["probs"].argmax(1) == z["y"]
            rows_dd.append(dict(seed=sd, n_all=len(ok), acc_all=100 * ok.mean(), n_no_dup=int(keep.sum()), acc_no_dup=100 * ok[keep].mean(),
                                n_dup=int((~keep).sum()), acc_dup=100 * ok[~keep].mean()))
    if rows_dd: pd.DataFrame(rows_dd).to_csv(f"{OUT_DIR}/dedup_accuracy.csv", index=False)

# %% FIGURES (seed SEEDS[0]) -> PNG files in OUT_DIR ---------------------------------
def make_figures():
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve, auc
    s0 = SEEDS[0]; hf = f"{OUT_DIR}/history_HEASNet_s{s0}.csv"; pf = f"{OUT_DIR}/preds_HEASNet_s{s0}.npz"
    if os.path.exists(hf):
        hh = pd.read_csv(hf); x = np.arange(1, len(hh) + 1)
        for key, lab, fn in (("loss", "Loss", "fig2_loss.png"), ("acc", "Accuracy", "fig3_accuracy.png")):
            fig, ax = plt.subplots(figsize=(6.4, 4.2), dpi=200)
            ax.plot(x, hh[f"train_{key}"], label="Training"); ax.plot(x, hh[f"val_{key}"], label="Validation")
            ax.axvline(HEAD_EPOCHS + 0.5, color="gray", ls=":"); ax.set_xlabel("Epoch"); ax.set_ylabel(lab); ax.grid(alpha=.25); ax.legend()
            fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{fn}"); plt.close(fig)
    if os.path.exists(pf):
        z = np.load(pf); P, Y = z["probs"], z["y"]
        cm = confusion_matrix(Y, P.argmax(1)); fig, ax = plt.subplots(figsize=(7.6, 6), dpi=200); im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(K)); ax.set_yticks(range(K)); ax.set_xticklabels(classes, rotation=60, ha="right", fontsize=7); ax.set_yticklabels(classes, fontsize=7)
        for i in range(K):
            for j in range(K):
                if cm[i, j]: ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > cm.max() / 2 else "black")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); fig.colorbar(im, fraction=.046); fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig4_confusion.png"); plt.close(fig)
        fig, ax = plt.subplots(figsize=(6.4, 5), dpi=200)
        for c in range(K):
            fpr, tpr, _ = roc_curve((Y == c).astype(int), P[:, c]); ax.plot(fpr, tpr, lw=1.2, label=f"{classes[c]} (AUC {auc(fpr, tpr):.3f})")
        ax.plot([0, 1], [0, 1], "k:", lw=1); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend(fontsize=6, loc="lower right")
        fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig5_roc.png"); plt.close(fig)
        print("figures written to", OUT_DIR)
if "main" in RUN or "eval" in RUN:
    try: make_figures()
    except Exception as e: print("figures skipped:", e)

# %% ONE BLOCK TO COPY BACK ---------------------------------------------------
def print_results_block():
    """Prints every result file in one block. Copy everything between the two ##### lines and send it back."""
    print("#" * 20, "COPY FROM HERE", "#" * 20)
    if QUICK: print("!!! QUICK MODE (1 epoch): do NOT use these numbers in the paper !!!")
    print(f"dataset={DATASET} processed={PROCESSED} aug_in_train={USE_AUG_IN_TRAIN} backbone=EfficientNetV2-{BACKBONE.upper()} window={WINDOW} seeds={SEEDS}")
    print(f"n_classes={K} n_train={len(tr)} n_val={len(va)} n_test={len(te)}")
    print("classes:", classes)
    for name in ["class_split_counts.csv", "test_metrics.json", "main_per_seed.csv", "test_classification_report.csv", "confusion_matrix.csv",
                 "ablation_results.csv", "baselines.csv", "mcnemar.csv", "sweeps.csv", "overlap_check.json", "dedup_accuracy.csv", "split_metrics.csv", "classification_report_train.csv", "classification_report_val.csv",
                 "confusion_matrix_train.csv", "confusion_matrix_val.csv", "bootstrap_ci_test.csv", "per_class_recall_ci_test.csv", "statistical_tests.json",
                 "mcnemar_vs_other_models.csv", "confidence_summary.json", "selective_accuracy_test.csv", "efficiency_full.json", "gradcam_attention_correlation.csv"]:
        f = f"{OUT_DIR}/{name}"
        if os.path.exists(f):
            print(f"\n=== {name} ==="); print(open(f).read().strip())
    h = f"{OUT_DIR}/history_HEASNet_s{SEEDS[0]}.csv"
    if os.path.exists(h):
        hh = pd.read_csv(h); b = hh.loc[hh.val_acc.idxmax()]
        print(f"\n=== training summary (seed {SEEDS[0]}) ===")
        print(f"best val acc {b.val_acc:.4f} at epoch {int(b.epoch)}; final epoch train acc {hh.train_acc.iloc[-1]:.4f}, val acc {hh.val_acc.iloc[-1]:.4f}, train loss {hh.train_loss.iloc[-1]:.4f}, val loss {hh.val_loss.iloc[-1]:.4f}")
        print("history.csv (every 5th epoch):"); print(hh.iloc[4::5].round(4).to_csv(index=False).strip())
    print("#" * 20, "COPY TO HERE", "#" * 20)

print("Done. Results in", OUT_DIR)
print_results_block()

In [ ]:
# ================================================================
# UNIFIED HEASNet EXPERIMENT: HyperKvasir + Kvasir-V2
# One pipeline | 40 epochs per dataset | clean split | full evaluation
# Run in Kaggle with GPU enabled and both datasets attached.
# ================================================================

import os, gc, json, time, random, warnings, zipfile, shutil, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_curve, auc, roc_auc_score, cohen_kappa_score,
    matthews_corrcoef, log_loss, f1_score
)
from scipy.stats import binomtest

# ----------------------------- CONFIG -----------------------------
SEED = 42
EPOCHS = 40
HEAD_WARMUP_EPOCHS = 5    # first 5 epochs: train projection/attention/head
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
LR_HEAD = 1e-4
LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-5
LABEL_SMOOTHING = 0.05
DROPOUT = 0.30
EMBED_DIM = 512
NUM_HEADS = 8
WINDOW_SIZE = 4
VAL_FRAC = 0.15
TEST_FRAC = 0.15
BOOTSTRAP_REPS = 1000
GRADCAM_PER_CLASS = 2
USE_PRETRAINED = True
PIN_MEMORY = torch.cuda.is_available()

# Exact mounted dataset paths confirmed from the Kaggle directory audit.
# HyperKvasir: train/<class>/ and test/<class>/
# Kvasir-V2: <class>/ directly under the dataset root.
DATASETS = {
    "HyperKvasir": {
        "root": "/kaggle/input/datasets/visheshtanwar26/processed-datatset",
        "layout": "predefined_train_test",
        "expected_classes": 12
    },
    "Kvasir-V2": {
        "root": "/kaggle/input/datasets/subhajeetdas/kvasir-v2-gastrointestinal-image-dataset/KVASIR V2",
        "layout": "class_folders",
        "expected_classes": 8
    }
}

from pathlib import Path

def find_dataset_folder(root_path):
    root = Path(root_path)

    if not root.exists():
        raise FileNotFoundError(
            f"Dataset path does not exist: {root}"
        )

    if not root.is_dir():
        raise NotADirectoryError(
            f"Expected a directory: {root}"
        )

    print(f"Dataset found: {root}")

    return root

OUTPUT_ROOT = "/kaggle/working/HEASNet_Both_Datasets_Results"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 90)
print("UNIFIED HEASNet EXPERIMENT")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print(f"Epochs={EPOCHS} | Image size={IMG_SIZE} | Batch={BATCH_SIZE} | Seed={SEED}")
print("=" * 90)

# ----------------------------- REPRODUCIBILITY -----------------------------
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything()

# ----------------------------- DATA DISCOVERY -----------------------------
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def normalize_class_name(name):
    return name.lower().strip().replace(" ", "-").replace("_", "-")

def collect_class_folder_images(split_root, split_name):
    """
    For HyperKvasir: expects train/<class>/image and test/<class>/image.
    Label is the first directory below train or test, even if images are nested.
    """
    split_root = Path(split_root)
    records = []
    for p in split_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            rel = p.relative_to(split_root)
            if len(rel.parts) < 2:
                continue
            label = normalize_class_name(rel.parts[0])
            records.append((str(p), label, split_name))
    return records

def discover_hyperkvasir(root):
    root = Path(root)
    train_dir, test_dir = root / "train", root / "test"
    if not train_dir.exists() or not test_dir.exists():
        raise FileNotFoundError(
            f"Expected HyperKvasir folders:\n{train_dir}\n{test_dir}\n"
            "The screenshot shows train/ and test/ under Hyper_Kasvir_dataset."
        )
    records = collect_class_folder_images(train_dir, "train")
    records += collect_class_folder_images(test_dir, "test")
    df = pd.DataFrame(records, columns=["path", "label", "source_split"]).drop_duplicates("path")
    if df.empty:
        raise RuntimeError("No HyperKvasir images found inside train/ and test/.")
    train_classes = set(df.loc[df.source_split == "train", "label"].unique())
    test_classes = set(df.loc[df.source_split == "test", "label"].unique())
    if train_classes != test_classes:
        print(
            "WARNING: the provided HyperKvasir train/test folders do not contain "
            "identical class sets. Missing classes will be shown as zero in the "
            "split audit; per-class test ROC-AUC may be undefined for absent classes. "
            f"Train-only: {sorted(train_classes-test_classes)}; "
            f"test-only: {sorted(test_classes-train_classes)}"
        )
    print(f"HyperKvasir: {len(df):,} images | {len(train_classes)} classes")
    print("Class counts by provided split:")
    print(pd.crosstab(df.label, df.source_split).to_string())
    return df.reset_index(drop=True)

def discover_kvasir_v2(root):
    root = Path(root)
    records = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            # In the Kvasir-V2 structure shown, each image is directly in one
            # of the eight class folders below the KVASIR V2 directory.
            rel = p.relative_to(root)
            if len(rel.parts) < 2:
                continue
            label = normalize_class_name(rel.parts[0])
            records.append((str(p), label))
    df = pd.DataFrame(records, columns=["path", "label"]).drop_duplicates("path")
    if df.empty:
        raise RuntimeError(f"No class-folder images found under {root}")
    print(f"Kvasir-V2: {len(df):,} images | {df.label.nunique()} classes")
    print("Per-class image counts:")
    print(df.label.value_counts().sort_index().to_string())
    return df.reset_index(drop=True)

def create_kvasir_splits(df):
    """Kvasir-V2: stratified 70/15/15 split at image level."""
    df = df.copy()
    classes = sorted(df.label.unique())
    class_to_idx = {name: i for i, name in enumerate(classes)}
    df["target"] = df.label.map(class_to_idx).astype(int)
    train_val, test_df = train_test_split(
        df, test_size=TEST_FRAC, stratify=df.target, random_state=SEED
    )
    relative_val = VAL_FRAC / (1.0 - TEST_FRAC)
    train_df, val_df = train_test_split(
        train_val, test_size=relative_val, stratify=train_val.target,
        random_state=SEED
    )
    return (classes, train_df.reset_index(drop=True),
            val_df.reset_index(drop=True), test_df.reset_index(drop=True))

def create_hyperkvasir_splits(df):
    """
    Preserve the supplied HyperKvasir test set.
    Split the supplied training set into train/validation while keeping an
    original image and its *_aug_* derivatives in the same partition.
    """
    from sklearn.model_selection import GroupShuffleSplit

    df = df.copy()
    classes = sorted(df.label.unique())
    class_to_idx = {name: i for i, name in enumerate(classes)}
    df["target"] = df.label.map(class_to_idx).astype(int)

    original_train = df[df.source_split == "train"].copy()
    test_df = df[df.source_split == "test"].copy()

    # Group augmented files with their source image.
    original_train["group_id"] = original_train["path"].map(
        lambda p: re.sub(r"_aug_.*$", "", Path(p).stem)
    )

    # Try deterministic group-shuffle candidates; retain all classes in both
    # train and validation and choose the split closest to VAL_FRAC.
    best = None
    best_score = float("inf")
    for seed in range(SEED, SEED + 100):
        splitter = GroupShuffleSplit(
            n_splits=1, test_size=VAL_FRAC, random_state=seed
        )
        train_idx, val_idx = next(splitter.split(
            original_train, original_train.target,
            groups=original_train.group_id
        ))
        candidate_train = original_train.iloc[train_idx].copy()
        candidate_val = original_train.iloc[val_idx].copy()

        if (set(candidate_train.label) != set(classes) or
                set(candidate_val.label) != set(classes)):
            continue

        score = abs(len(candidate_val) / len(original_train) - VAL_FRAC)
        if score < best_score:
            best_score = score
            best = (candidate_train, candidate_val)

    if best is None:
        raise RuntimeError(
            "Could not create a group-aware HyperKvasir validation split "
            "containing every class. Inspect class counts and augmentation "
            "filename patterns. A random image-level fallback is intentionally "
            "not used because it could leak augmented siblings."
        )

    train_df, val_df = best
    if set(train_df.group_id) & set(val_df.group_id):
        raise RuntimeError("Augmentation-group leakage detected.")

    return (
        classes,
        train_df.drop(columns=["group_id"]).reset_index(drop=True),
        val_df.drop(columns=["group_id"]).reset_index(drop=True),
        test_df.reset_index(drop=True)
    )


# ----------------------------- DATA QUALITY + SPLIT AUDIT -----------------------------
def verify_images_and_remove_corrupt(df, dataset_name, out_dir):
    """Verify image readability before splitting/training and log rejected files."""
    good, bad = [], []
    for row in df.itertuples(index=False):
        try:
            with Image.open(row.path) as im:
                im.verify()
            good.append(row.Index if hasattr(row, "Index") else row.path)
        except Exception as exc:
            bad.append({"path": row.path, "label": row.label, "error": repr(exc)})

    if bad:
        bad_df = pd.DataFrame(bad)
        bad_df.to_csv(os.path.join(out_dir, "corrupt_or_unreadable_images.csv"), index=False)
        bad_paths = set(bad_df.path)
        df = df[~df.path.isin(bad_paths)].copy()
        print(f"Removed {len(bad)} corrupt/unreadable files from {dataset_name}.")
    else:
        pd.DataFrame(columns=["path", "label", "error"]).to_csv(
            os.path.join(out_dir, "corrupt_or_unreadable_images.csv"), index=False
        )
        print("Image integrity check: all discovered files are readable.")
    if df.empty:
        raise RuntimeError(f"No readable images remain for {dataset_name}.")
    return df.reset_index(drop=True)

def build_split_audit(classes, train_df, val_df, test_df, out_dir):
    """Save full class-by-split counts, percentages and integrity checks."""
    frames = {"train": train_df, "validation": val_df, "test": test_df}
    count_table = pd.DataFrame(index=classes)
    for split_name, frame in frames.items():
        counts = frame.label.value_counts()
        count_table[split_name] = [int(counts.get(c, 0)) for c in classes]
    count_table.index.name = "class"
    count_table["total"] = count_table.sum(axis=1)
    count_table.to_csv(os.path.join(out_dir, "CLASS_COUNTS_EVERY_SPLIT.csv"))

    percent_table = count_table[["train", "validation", "test"]].div(
        count_table[["train", "validation", "test"]].sum(axis=0), axis=1
    ) * 100
    percent_table.index.name = "class"
    percent_table.to_csv(os.path.join(out_dir, "CLASS_PERCENTAGES_WITHIN_SPLIT.csv"))

    summary_rows = []
    for split_name, frame in frames.items():
        summary_rows.append({
            "split": split_name,
            "images": int(len(frame)),
            "percentage_of_dataset": float(100 * len(frame) /
                (len(train_df) + len(val_df) + len(test_df))),
            "number_of_classes_present": int(frame.label.nunique()),
            "missing_classes": "; ".join([c for c in classes if c not in set(frame.label)])
        })
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(os.path.join(out_dir, "SPLIT_SIZE_AND_CLASS_AUDIT.csv"), index=False)

    # A single image path must occur in exactly one split.
    path_sets = {k: set(v.path) for k, v in frames.items()}
    overlaps = {
        "train_validation": len(path_sets["train"] & path_sets["validation"]),
        "train_test": len(path_sets["train"] & path_sets["test"]),
        "validation_test": len(path_sets["validation"] & path_sets["test"])
    }
    all_paths = list(path_sets["train"] | path_sets["validation"] | path_sets["test"])
    duplicate_paths = len(all_paths) - sum(len(x) for x in path_sets.values())
    audit = {
        "class_names": classes,
        "number_of_classes": len(classes),
        "total_images_after_integrity_check": len(all_paths),
        "split_sizes": {k: len(v) for k, v in frames.items()},
        "overlap_counts": overlaps,
        "duplicate_path_count_across_splits": duplicate_paths,
        "all_classes_present_in_training": all(c in set(train_df.label) for c in classes),
        "all_classes_present_in_validation": all(c in set(val_df.label) for c in classes),
        "all_classes_present_in_test": all(c in set(test_df.label) for c in classes),
        "note": "Missing test classes are reported, not silently fabricated. Their class-wise sensitivity/AUC cannot be estimated when no test examples exist."
    }
    with open(os.path.join(out_dir, "SPLIT_INTEGRITY_AUDIT.json"), "w") as f:
        json.dump(audit, f, indent=2)

    print("\nCOMPLETE CLASS COUNTS (zero means no images for that class in that split)")
    print(count_table.to_string())
    print("\nSPLIT SIZE / CLASS COVERAGE")
    print(summary.to_string(index=False))
    print("\nSPLIT OVERLAP AUDIT:", overlaps)
    if any(overlaps.values()) or duplicate_paths:
        raise RuntimeError("Split leakage detected: image paths overlap across splits.")
    if not audit["all_classes_present_in_training"]:
        raise RuntimeError("At least one class is missing from training. Training stopped.")
    if not audit["all_classes_present_in_validation"]:
        raise RuntimeError(
            "At least one class is missing from validation. "
            "Cannot reliably select the best checkpoint; inspect class counts."
        )
    return count_table, summary, audit

# ----------------------------- TRANSFORMS / DATASET -----------------------------
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_transform = train_transform

class ImageDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame.path.tolist()
        self.targets = frame.target.astype(int).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        with Image.open(path) as im:
            image = im.convert("RGB")
        image = self.transform(image)
        return image, self.targets[idx]

def make_loaders(train_df, val_df, test_df):
    train_ds = ImageDataset(train_df, train_transform)
    val_ds = ImageDataset(val_df, eval_transform)
    test_ds = ImageDataset(test_df, eval_transform)
    generator = torch.Generator().manual_seed(SEED)
    common = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                  pin_memory=PIN_MEMORY, persistent_workers=(NUM_WORKERS > 0))
    train_loader = DataLoader(train_ds, shuffle=True, generator=generator, **common)
    val_loader = DataLoader(val_ds, shuffle=False, **common)
    test_loader = DataLoader(test_ds, shuffle=False, **common)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds

# ----------------------------- HEASNet MODEL -----------------------------
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x):
        z = x.mean(dim=(2, 3))
        z = torch.sigmoid(self.fc2(F.relu(self.fc1(z))))
        return x * z[:, :, None, None]

class WindowAttention(nn.Module):
    def __init__(self, dim, heads=8, window=4):
        super().__init__()
        self.window = window
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        batch, channels, height, width = x.shape
        w = self.window
        pad_h, pad_w = (-height) % w, (-width) % w
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h))
        hp, wp = height + pad_h, width + pad_w
        tokens = (x.permute(0, 2, 3, 1)
                    .reshape(batch, hp // w, w, wp // w, w, channels)
                    .permute(0, 1, 3, 2, 4, 5)
                    .reshape(-1, w * w, channels))
        attended = self.attn(tokens, tokens, tokens, need_weights=False)[0]
        tokens = self.norm(tokens + attended)
        x = (tokens.reshape(batch, hp // w, wp // w, w, w, channels)
                   .permute(0, 1, 3, 2, 4, 5)
                   .reshape(batch, hp, wp, channels)
                   .permute(0, 3, 1, 2))
        return x[:, :, :height, :width]

class AxialAttentionHeight(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.permute(0, 3, 2, 1).reshape(batch * width, height, channels)
        tokens = self.norm(tokens + self.attn(tokens, tokens, tokens, need_weights=False)[0])
        return tokens.reshape(batch, width, height, channels).permute(0, 3, 2, 1)

class HEASNet(nn.Module):
    """
    HEASNet = EfficientNetV2-S feature extractor + projection + SE +
    local window attention + height-axis attention + classification head.
    """
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_V2_S_Weights.DEFAULT if pretrained else None
        self.backbone = models.efficientnet_v2_s(weights=weights).features
        self.projection = nn.Conv2d(1280, EMBED_DIM, kernel_size=1, bias=False)
        self.se = SqueezeExcitation(EMBED_DIM)
        self.window_attention = WindowAttention(EMBED_DIM, NUM_HEADS, WINDOW_SIZE)
        self.axial_attention = AxialAttentionHeight(EMBED_DIM, NUM_HEADS)
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(EMBED_DIM, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        x = self.projection(features)
        x = self.se(x)
        x = self.window_attention(x)
        x = self.axial_attention(x)
        x = x.mean(dim=(2, 3))
        return self.classifier(self.dropout(x))

    def head_parameters(self):
        modules = [self.projection, self.se, self.window_attention,
                   self.axial_attention, self.classifier]
        return [p for module in modules for p in module.parameters()]

# ----------------------------- TRAINING -----------------------------
def run_train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

def train_heasnet(model, train_loader, val_loader, out_dir):
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    history = []
    best_val_acc = -1.0
    best_epoch = 0
    best_path = os.path.join(out_dir, "best_HEASNet.pt")

    # Stage 1: freeze pretrained backbone, train the new HEASNet layers.
    for p in model.backbone.parameters():
        p.requires_grad = False
    head_optimizer = torch.optim.AdamW(
        model.head_parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY
    )

    # Stage 2: unfreeze all layers; use a smaller LR for the backbone.
    fine_optimizer = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
        {"params": model.head_parameters(), "lr": LR_HEAD}
    ], weight_decay=WEIGHT_DECAY)

    for epoch in range(1, EPOCHS + 1):
        if epoch == HEAD_WARMUP_EPOCHS + 1:
            for p in model.backbone.parameters():
                p.requires_grad = True

        optimizer = head_optimizer if epoch <= HEAD_WARMUP_EPOCHS else fine_optimizer
        start = time.time()
        train_loss, train_acc = run_train_epoch(
            model, train_loader, criterion, optimizer, scaler
        )
        val_loss, val_acc = evaluate_epoch(model, val_loader, criterion)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        epoch_seconds = time.time() - start
        history.append({
            "epoch_seconds": epoch_seconds,
            "epoch": epoch, "train_loss": train_loss, "train_accuracy": train_acc,
            "val_loss": val_loss, "val_accuracy": val_acc,
            "learning_rate_head": LR_HEAD,
            "learning_rate_backbone": 0.0 if epoch <= HEAD_WARMUP_EPOCHS else LR_BACKBONE
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            torch.save(model.state_dict(), best_path)

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"Val loss {val_loss:.4f} acc {val_acc:.4f} | "
            f"Best val {best_val_acc:.4f} (epoch {best_epoch}) | "
            f"{epoch_seconds:.1f}s"
        )

    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
    pd.DataFrame(history).to_csv(os.path.join(out_dir, "training_history.csv"), index=False)
    return model, pd.DataFrame(history), best_epoch, best_val_acc, best_path

# ----------------------------- PREDICTION / METRICS -----------------------------
@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs, all_y = [], []
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(images)
        all_probs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
        all_y.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_y)

def expected_calibration_error(probs, y, bins=15):
    confidence = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correct = (predictions == y).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)

def wilson_interval(successes, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = successes / n
    den = 1 + z*z/n
    center = (p + z*z/(2*n)) / den
    half = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return float(center-half), float(center+half)

def bootstrap_metric_ci(y, probs, metric_fn, reps=BOOTSTRAP_REPS, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    values = []
    for _ in range(reps):
        idx = rng.integers(0, n, n)
        try:
            value = metric_fn(y[idx], probs[idx])
            if np.isfinite(value):
                values.append(value)
        except Exception:
            pass
    if not values:
        return np.nan, np.nan
    return tuple(np.percentile(values, [2.5, 97.5]).astype(float))

def compute_metrics(y, probs, classes):
    n_classes = len(classes)
    pred = probs.argmax(axis=1)
    precision, recall, f1, support = precision_recall_fscore_support(
        y, pred, labels=np.arange(n_classes), zero_division=0
    )
    cm = confusion_matrix(y, pred, labels=np.arange(n_classes))
    specificity = []
    for k in range(n_classes):
        tp = cm[k, k]
        fn = cm[k, :].sum() - tp
        fp = cm[:, k].sum() - tp
        tn = cm.sum() - tp - fn - fp
        specificity.append(tn / (tn + fp) if (tn + fp) else np.nan)

    per_class_auc = []
    y_bin = np.eye(n_classes)[y]
    for k in range(n_classes):
        try:
            per_class_auc.append(roc_auc_score(y_bin[:, k], probs[:, k]))
        except ValueError:
            per_class_auc.append(np.nan)

    macro_auc = float(np.nanmean(per_class_auc)) if np.isfinite(per_class_auc).any() else np.nan
    try:
        micro_auc = float(roc_auc_score(y_bin.ravel(), probs.ravel()))
    except ValueError:
        micro_auc = np.nan

    metrics = {
        "n": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "macro_precision": float(np.mean(precision)),
        "macro_recall": float(np.mean(recall)),
        "macro_f1": float(np.mean(f1)),
        "weighted_f1": float(f1_score(y, pred, average="weighted", zero_division=0)),
        "cohen_kappa": float(cohen_kappa_score(y, pred)),
        "mcc": float(matthews_corrcoef(y, pred)),
        "macro_roc_auc_ovr": macro_auc,
        "micro_roc_auc": micro_auc,
        "log_loss": float(log_loss(y, probs, labels=np.arange(n_classes))),
        "brier_score": float(np.mean(np.sum((probs - y_bin) ** 2, axis=1))),
        "ece_15_bins": expected_calibration_error(probs, y)
    }
    per_class = pd.DataFrame({
        "class": classes, "precision": precision, "recall_sensitivity": recall,
        "specificity": specificity, "f1_score": f1, "support": support,
        "roc_auc_ovr": per_class_auc
    })
    return metrics, per_class, cm

# ----------------------------- PLOTS -----------------------------
def save_training_curves(history, out_dir, dataset_name):
    epochs = history.epoch
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history.train_loss, label="Train loss")
    ax.plot(epochs, history.val_loss, label="Validation loss")
    ax.set(title=f"{dataset_name}: Training and validation loss",
           xlabel="Epoch", ylabel="Cross-entropy loss")
    ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "01_loss_curve.png"), dpi=220); plt.show(); plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history.train_accuracy, label="Train accuracy")
    ax.plot(epochs, history.val_accuracy, label="Validation accuracy")
    ax.set(title=f"{dataset_name}: Training and validation accuracy",
           xlabel="Epoch", ylabel="Accuracy")
    ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "02_accuracy_curve.png"), dpi=220); plt.show(); plt.close(fig)

def save_confusion_matrix(cm, classes, out_dir, normalized=False):
    matrix = cm.astype(float)
    if normalized:
        row_sums = matrix.sum(axis=1, keepdims=True)
        matrix = np.divide(matrix, row_sums, out=np.zeros_like(matrix), where=row_sums != 0)
    fig, ax = plt.subplots(figsize=(max(8, len(classes)*0.85), max(7, len(classes)*0.75)))
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=60, ha="right", fontsize=8)
    ax.set_yticklabels(classes, fontsize=8)
    threshold = np.nanmax(matrix) / 2 if matrix.size else 0
    for i in range(len(classes)):
        for j in range(len(classes)):
            value = f"{matrix[i,j]:.2f}" if normalized else f"{int(cm[i,j])}"
            ax.text(j, i, value, ha="center", va="center",
                    color="white" if matrix[i,j] > threshold else "black", fontsize=7)
    ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
    ax.set_title("Normalized confusion matrix" if normalized else "Confusion matrix (counts)")
    fig.colorbar(im, ax=ax, fraction=.046); fig.tight_layout()
    filename = "04_confusion_normalized.png" if normalized else "03_confusion_counts.png"
    fig.savefig(os.path.join(out_dir, filename), dpi=220); plt.show(); plt.close(fig)

def save_roc_curves(y, probs, classes, out_dir, split_name="test"):
    y_bin = np.eye(len(classes))[y]
    fig, ax = plt.subplots(figsize=(9, 7))
    auc_rows = []
    for k, name in enumerate(classes):
        if len(np.unique(y_bin[:, k])) < 2:
            auc_rows.append({
                "class": name, "auc_ovr": np.nan,
                "note": "Undefined: class absent or no negative examples"
            })
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, k], probs[:, k])
        class_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=1.5, label=f"{name} (AUC={class_auc:.3f})")
        auc_rows.append({"class": name, "auc_ovr": class_auc, "note": ""})
    if len(np.unique(y_bin.ravel())) == 2:
        fpr_micro, tpr_micro, _ = roc_curve(y_bin.ravel(), probs.ravel())
        micro_auc = auc(fpr_micro, tpr_micro)
        ax.plot(fpr_micro, tpr_micro, "k--", lw=2,
                label=f"Micro-average (AUC={micro_auc:.3f})")
    ax.plot([0, 1], [0, 1], "k:", label="Chance")
    ax.set(xlim=(0,1), ylim=(0,1.02), xlabel="False positive rate",
           ylabel="True positive rate", title=f"{split_name.title()} one-vs-rest ROC curves")
    ax.legend(loc="lower right", fontsize=7); ax.grid(alpha=.2); fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"05_{split_name}_classwise_ROC_AUC.png"), dpi=220)
    plt.show(); plt.close(fig)
    pd.DataFrame(auc_rows).to_csv(os.path.join(out_dir, f"{split_name}_classwise_AUC.csv"), index=False)

def save_classwise_metric_plot(per_class, out_dir):
    cols = ["precision", "recall_sensitivity", "specificity", "f1_score", "roc_auc_ovr"]
    plot_df = per_class.set_index("class")[cols]
    ax = plot_df.plot(kind="bar", figsize=(max(10, len(plot_df)*0.9), 6))
    ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
    ax.set_title("Class-wise classification metrics")
    ax.grid(axis="y", alpha=.25); plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, "06_classwise_metrics.png"), dpi=220)
    plt.show(); plt.close()

def save_reliability_plot(y, probs, out_dir):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == y).astype(float)
    edges = np.linspace(0, 1, 16)
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.any():
            xs.append(conf[mask].mean()); ys.append(correct[mask].mean())
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0,1], [0,1], "k:", label="Perfect calibration")
    ax.plot(xs, ys, "o-", label="HEASNet")
    ax.set(xlabel="Mean predicted confidence", ylabel="Observed accuracy",
           title="Test-set reliability diagram", xlim=(0,1), ylim=(0,1))
    ax.grid(alpha=.2); ax.legend(); fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "07_reliability_diagram.png"), dpi=220)
    plt.show(); plt.close(fig)

# ----------------------------- GRAD-CAM EXPLAINABILITY -----------------------------
def save_gradcam_examples(model, test_dataset, test_probs, test_y, classes, out_dir):
    """
    Grad-CAM on the final EfficientNetV2-S feature map.
    These are model-attribution heatmaps, not segmentation masks or proof of causality.
    """
    target_layer = model.backbone[-1]
    activations, gradients = {}, {}

    def forward_hook(module, inputs, output):
        activations["value"] = output

    def backward_hook(module, grad_input, grad_output):
        gradients["value"] = grad_output[0]

    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_full_backward_hook(backward_hook)
    model.eval()
    selected = []
    pred = test_probs.argmax(axis=1)
    for class_idx in range(len(classes)):
        candidates = np.where(test_y == class_idx)[0]
        if len(candidates):
            # Include a correct example first when possible.
            correct = [i for i in candidates if pred[i] == class_idx]
            wrong = [i for i in candidates if pred[i] != class_idx]
            selected.extend((correct[:GRADCAM_PER_CLASS] or wrong[:1]))

    if not selected:
        h1.remove(); h2.remove()
        return

    cols = 4
    rows = int(np.ceil(len(selected) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3.5), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")

    for pos, idx in enumerate(selected):
        image_tensor, true_label = test_dataset[idx]
        x = image_tensor.unsqueeze(0).to(DEVICE)
        model.zero_grad(set_to_none=True)
        logits = model(x)
        predicted = int(logits.argmax(1).item())
        logits[0, predicted].backward()

        act = activations["value"].detach()
        grad = gradients["value"].detach()
        weights = grad.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * act).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        img = Image.open(test_dataset.paths[idx]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        ax = axes[pos // cols, pos % cols]
        ax.imshow(img)
        ax.imshow(cam, cmap="jet", alpha=0.40)
        ax.set_title(
            f"True: {classes[true_label]}\nPred: {classes[predicted]}",
            fontsize=8, color=("green" if predicted == true_label else "red")
        )
        ax.axis("off")

    h1.remove(); h2.remove()
    fig.suptitle("HEASNet Grad-CAM explanations (test examples)", fontsize=13)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "08_gradcam_explainability.png"), dpi=220)
    plt.show(); plt.close(fig)

# ----------------------------- FULL REPORT -----------------------------
def evaluate_and_report(model, loaders, datasets, classes, out_dir, train_majority_class):
    split_results = {}
    for split_name, loader in zip(["train", "validation", "test"], loaders):
        probs, y = predict(model, loader)
        metrics, per_class, cm = compute_metrics(y, probs, classes)
        split_results[split_name] = {"probs": probs, "y": y, "metrics": metrics,
                                     "per_class": per_class, "cm": cm}
        per_class.to_csv(os.path.join(out_dir, f"{split_name}_classwise_metrics.csv"), index=False)
        pd.DataFrame(cm, index=classes, columns=classes).to_csv(
            os.path.join(out_dir, f"{split_name}_confusion_matrix.csv")
        )
        print(f"\n{split_name.upper()} METRICS")
        print(pd.Series(metrics).to_string())
        print("\nClass-wise metrics:")
        print(per_class.round(4).to_string(index=False))

    # Unified split summary.
    summary = pd.DataFrame([
        {"split": name, **result["metrics"]}
        for name, result in split_results.items()
    ])
    summary.to_csv(os.path.join(out_dir, "all_split_metrics.csv"), index=False)

    # Test-only statistical inference: test data were not used for model selection.
    y = split_results["test"]["y"]
    probs = split_results["test"]["probs"]
    pred = probs.argmax(axis=1)
    test_metrics = split_results["test"]["metrics"]
    n_classes = len(classes)
    majority_rate = Counter(y.tolist()).most_common(1)[0][1] / len(y)
    baseline_pred = np.full_like(y, fill_value=int(train_majority_class))
    model_correct = (pred == y)
    baseline_correct = (baseline_pred == y)
    only_model_correct = int(np.sum(model_correct & ~baseline_correct))
    only_baseline_correct = int(np.sum(~model_correct & baseline_correct))
    discordant = only_model_correct + only_baseline_correct
    mcnemar_p = float(binomtest(only_model_correct, discordant, 0.5).pvalue) if discordant else 1.0
    mcnemar_result = {
        "baseline": "constant class predicted from training-set majority",
        "training_majority_class": str(classes[int(train_majority_class)]),
        "only_HEASNet_correct": only_model_correct,
        "only_baseline_correct": only_baseline_correct,
        "discordant_pairs": discordant,
        "exact_McNemar_binomial_p": mcnemar_p
    }
    pd.DataFrame([mcnemar_result]).to_csv(
        os.path.join(out_dir, "test_McNemar_vs_train_majority_baseline.csv"), index=False
    )

    acc_ci = bootstrap_metric_ci(y, probs, lambda yy, pp: accuracy_score(yy, pp.argmax(1)))
    f1_ci = bootstrap_metric_ci(y, probs, lambda yy, pp: f1_score(
        yy, pp.argmax(1), average="macro", zero_division=0))
    auc_ci = bootstrap_metric_ci(y, probs, lambda yy, pp: roc_auc_score(
        yy, pp, multi_class="ovr", average="macro", labels=np.arange(n_classes)))

    chance_test = binomtest(int((pred == y).sum()), len(y), 1/n_classes, alternative="greater")
    majority_test = binomtest(int((pred == y).sum()), len(y), majority_rate, alternative="greater")

    recall_ci_rows = []
    for _, row in split_results["test"]["per_class"].iterrows():
        lo, hi = wilson_interval(int(round(row.recall_sensitivity * row.support)), int(row.support))
        recall_ci_rows.append({
            "class": row["class"], "recall": row.recall_sensitivity,
            "recall_CI95_low": lo, "recall_CI95_high": hi, "support": int(row.support)
        })
    pd.DataFrame(recall_ci_rows).to_csv(
        os.path.join(out_dir, "test_classwise_recall_95CI.csv"), index=False
    )

    statistical = {
        "test_accuracy_95CI_bootstrap": acc_ci,
        "test_macro_F1_95CI_bootstrap": f1_ci,
        "test_macro_ROC_AUC_95CI_bootstrap": auc_ci,
        "binomial_test_vs_uniform_chance_p": float(chance_test.pvalue),
        "binomial_test_vs_test_majority_rate_p": float(majority_test.pvalue),
        "test_majority_class_rate": float(majority_rate),
        "McNemar_vs_training_majority_baseline": mcnemar_result,
        "note": "Bootstrap confidence intervals are percentile CIs. Binomial tests are one-sided and are approximate inferential comparisons; report dataset/sampling limitations."
    }
    with open(os.path.join(out_dir, "statistical_tests.json"), "w") as f:
        json.dump(statistical, f, indent=2)
    print("\nTEST STATISTICAL ANALYSIS")
    print(json.dumps(statistical, indent=2))

    # Test classification report with all classes in fixed order.
    report = classification_report(
        y, pred, labels=np.arange(n_classes), target_names=classes,
        output_dict=True, zero_division=0
    )
    pd.DataFrame(report).transpose().to_csv(
        os.path.join(out_dir, "test_classification_report_full.csv")
    )

    # Figures.
    save_confusion_matrix(split_results["test"]["cm"], classes, out_dir, normalized=False)
    save_confusion_matrix(split_results["test"]["cm"], classes, out_dir, normalized=True)
    save_roc_curves(y, probs, classes, out_dir, split_name="test")
    save_classwise_metric_plot(split_results["test"]["per_class"], out_dir)
    save_reliability_plot(y, probs, out_dir)
    save_gradcam_examples(model, datasets[2], probs, y, classes, out_dir)

    # Save predictions for reproducibility and later paired statistical comparisons.
    np.savez_compressed(
        os.path.join(out_dir, "test_predictions.npz"),
        y_true=y, probabilities=probs, y_pred=pred,
        class_names=np.asarray(classes)
    )
    return split_results, statistical

# ----------------------------- COMPLEXITY AND TIMING -----------------------------
def compute_complexity(model, out_dir):
    """Profile one 224x224 image. FLOPs conventions vary by profiler."""
    model = model.to(DEVICE).eval()
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    result = {"parameters_total": int(total), "parameters_trainable": int(trainable),
              "parameters_million": total / 1e6,
              "weights_fp32_MB_estimate": total * 4 / 1e6,
              "input_shape": [1, 3, IMG_SIZE, IMG_SIZE],
              "gmacs": None, "gflops": None,
              "profiling_note": "FLOPs/MACs depend on profiler support for attention operations."}
    try:
        from fvcore.nn import FlopCountAnalysis
        x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
        with torch.no_grad():
            flops = FlopCountAnalysis(model, x)
            flops.unsupported_ops_warnings(False)
            flops.uncalled_modules_warnings(False)
            result["gflops"] = float(flops.total() / 1e9)
            result["gmacs"] = float(flops.total() / 1e9)
            result["profiling_note"] = ("fvcore operation-count convention; GMACs is "
                "an approximation and not an independently validated count. "
                f"Unsupported ops: {dict(flops.unsupported_ops())}")
    except Exception as exc:
        result["profiling_note"] = ("FLOP profiling unavailable: " + repr(exc) +
            ". Install fvcore or use an attention-compatible profiler; do not report zero.")
    with open(os.path.join(out_dir, "model_complexity.json"), "w") as f:
        json.dump(result, f, indent=2)
    print("MODEL COMPLEXITY:", json.dumps(result, indent=2))
    return result

def benchmark_inference(model, out_dir, repeats=50, warmup=10):
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode():
        for _ in range(warmup):
            model(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        times = []
        for _ in range(repeats):
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(x)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            times.append((time.perf_counter()-t0)*1000)
    mean_ms = float(np.mean(times))
    result = {"batch_size": 1, "warmup_iterations": warmup,
              "timed_iterations": repeats, "mean_latency_ms_per_image": mean_ms,
              "median_latency_ms_per_image": float(np.median(times)),
              "throughput_images_per_second_batch1": 1000/mean_ms,
              "peak_gpu_allocated_MB_inference": (
                  float(torch.cuda.max_memory_allocated()/1e6) if DEVICE.type == "cuda" else None),
              "device": str(DEVICE)}
    with open(os.path.join(out_dir, "inference_benchmark.json"), "w") as f:
        json.dump(result, f, indent=2)
    return result

# ----------------------------- ONE DATASET EXPERIMENT -----------------------------
def run_dataset(dataset_name, config):
    print("\n\n" + "#" * 100)
    print(f"STARTING DATASET: {dataset_name}")
    print("#" * 100)
    seed_everything(SEED)

    out_dir = os.path.join(OUTPUT_ROOT, dataset_name.replace("-", "_"))
    os.makedirs(out_dir, exist_ok=True)

    root_path = config.get("root", config.get("folder_name"))
    if root_path is None:
        raise KeyError(
            f"No dataset path configured for {dataset_name}. "
            "Expected config key 'root' (or legacy 'folder_name')."
        )
    root = find_dataset_folder(root_path)
    if config["layout"] == "predefined_train_test":
        df = discover_hyperkvasir(root)
        if df.label.nunique() != config["expected_classes"]:
            raise RuntimeError(
                f"HyperKvasir expected {config['expected_classes']} classes, "
                f"but found {df.label.nunique()}: {sorted(df.label.unique())}"
            )
        split_method = "Provided train/test folders; validation carved from train only"
    elif config["layout"] == "class_folders":
        df = discover_kvasir_v2(root)
        if df.label.nunique() != config["expected_classes"]:
            raise RuntimeError(
                f"Kvasir-V2 expected {config['expected_classes']} classes, "
                f"but found {df.label.nunique()}: {sorted(df.label.unique())}"
            )
        split_method = "Stratified image-level 70/15/15 split"
    else:
        raise ValueError(f"Unknown dataset layout: {config['layout']}")
    # Verify all files can be opened before creating DataLoaders.
    df = verify_images_and_remove_corrupt(df, dataset_name, out_dir)

    # Recreate the split after removing unreadable files so counts stay accurate.
    if config["layout"] == "predefined_train_test":
        classes, train_df, val_df, test_df = create_hyperkvasir_splits(df)
    else:
        classes, train_df, val_df, test_df = create_kvasir_splits(df)

    print("\nDATA SPLIT SUMMARY")
    print("Split protocol:", split_method)
    print(f"Total: {len(df):,} | Train: {len(train_df):,} ({len(train_df)/len(df):.1%}) | "
          f"Validation: {len(val_df):,} ({len(val_df)/len(df):.1%}) | "
          f"Test: {len(test_df):,} ({len(test_df)/len(df):.1%})")
    split_df = pd.concat([
        train_df.assign(split="train"),
        val_df.assign(split="validation"),
        test_df.assign(split="test")
    ], ignore_index=True)
    split_df.to_csv(os.path.join(out_dir, "exact_image_split_manifest.csv"), index=False)
    counts, split_summary, split_audit = build_split_audit(
        classes, train_df, val_df, test_df, out_dir
    )

    train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = make_loaders(
        train_df, val_df, test_df
    )
    model = HEASNet(len(classes), pretrained=USE_PRETRAINED)
    print(f"\nModel: HEASNet | Classes: {len(classes)} | Parameters: "
          f"{sum(p.numel() for p in model.parameters())/1e6:.3f} million")
    print(
        f"Training protocol: {HEAD_WARMUP_EPOCHS} head warm-up epochs + "
        f"{max(0, EPOCHS - HEAD_WARMUP_EPOCHS)} fine-tuning epochs "
        f"= {EPOCHS} total epochs."
    )

    complexity = compute_complexity(model, out_dir)
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    model, history, best_epoch, best_val_acc, best_path = train_heasnet(
        model, train_loader, val_loader, out_dir
    )
    training_timing = {
        "total_training_and_validation_seconds": float(history.epoch_seconds.sum()),
        "mean_epoch_seconds": float(history.epoch_seconds.mean()),
        "peak_gpu_allocated_MB_training": (
            float(torch.cuda.max_memory_allocated()/1e6) if DEVICE.type == "cuda" else None)
    }
    with open(os.path.join(out_dir, "training_timing.json"), "w") as f:
        json.dump(training_timing, f, indent=2)
    inference_benchmark = benchmark_inference(model, out_dir)
    save_training_curves(history, out_dir, dataset_name)

    loaders = [DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          persistent_workers=(NUM_WORKERS > 0))
               for ds in [train_ds, val_ds, test_ds]]
    datasets = [train_ds, val_ds, test_ds]
    train_majority_class = int(train_df.target.value_counts().idxmax())
    split_results, statistical = evaluate_and_report(
        model, loaders, datasets, classes, out_dir, train_majority_class
    )

    # Final concise report for this dataset.
    final = {
        "dataset": dataset_name,
        "complexity": complexity,
        "training_timing": training_timing,
        "inference_benchmark": inference_benchmark,
        "num_images": int(len(df)),
        "num_classes": int(len(classes)),
        "classes": classes,
        "train_images": int(len(train_df)),
        "validation_images": int(len(val_df)),
        "test_images": int(len(test_df)),
        "epochs_completed": int(len(history)),
        "best_validation_epoch": int(best_epoch),
        "best_validation_accuracy": float(best_val_acc),
        "test_metrics": split_results["test"]["metrics"],
        "statistical_tests": statistical,
        "best_checkpoint": best_path
    }
    with open(os.path.join(out_dir, "FINAL_EXPERIMENT_SUMMARY.json"), "w") as f:
        json.dump(final, f, indent=2)
    print("\n" + "=" * 80)
    print(f"FINAL RESULT: {dataset_name}")
    print("=" * 80)
    print(pd.Series(final["test_metrics"]).round(5).to_string())
    print(f"Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}")
    print("All outputs saved to:", out_dir)

    del model, train_loader, val_loader, test_loader, loaders
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return final

# ----------------------------- RUN BOTH DATASETS -----------------------------
all_summaries = []
for dataset_name, config in DATASETS.items():
    try:
        result = run_dataset(dataset_name, config)
        all_summaries.append(result)
    except Exception as exc:
        print(f"\nERROR while processing {dataset_name}: {repr(exc)}")
        raise

# Combined comparison table (metrics are dataset-specific; do not interpret as a paired test).
comparison_rows = []
for item in all_summaries:
    comparison_rows.append({
        "dataset": item["dataset"],
        "images": item["num_images"],
        "classes": item["num_classes"],
        "train": item["train_images"],
        "validation": item["validation_images"],
        "test": item["test_images"],
        "best_epoch": item["best_validation_epoch"],
        "best_val_accuracy": item["best_validation_accuracy"],
        **item["test_metrics"]
    })
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(os.path.join(OUTPUT_ROOT, "BOTH_DATASETS_COMPARISON.csv"), index=False)
with open(os.path.join(OUTPUT_ROOT, "BOTH_DATASETS_FINAL_SUMMARY.json"), "w") as f:
    json.dump(all_summaries, f, indent=2)

print("\n\n" + "#" * 100)
print("FINAL COMPARISON: BOTH DATASETS")
print("#" * 100)
print(comparison.round(4).to_string(index=False))



